## Participants
- Eder Tarifa Fernández
- Zakaria Lasry Sahraoui

Grupo K

## Imports

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.model_selection import train_test_split
import math

## Data

In [ ]:
train = pd.read_csv('data3/test_reviews.csv')
train.head()

,review_id,user_id,business_id,target,useful,funny,cool,useful_user,funny_user,cool_user,...,cat_Event_Planning_&_Ser,cat_American_(Traditiona,cat_Sandwiches,cat_Active_Life,cat_Pizza,cat_Coffee_&_Tea,cat_Fast_Food,cat_American_(New),cat_Breakfast_&_Brunch,cat_Hotels_&_Travel
0,ZZO43qKB-s65zplC8RfJqw,-1BSu2dt_rOAqllw9ZDXtA,smkZq4G1AOm4V6p3id5sww,5.0,0,0,0,7.0,3.0,0.0,...,0,0,0,0,0,0,0,0,0,0
1,vojXOF_VOgvuKD95gCO8_Q,xpe178ng_gj5X6HgqtOing,96_c_7twb7hYRZ9HHrq01g,1.0,2,0,1,37.0,1.0,2.0,...,0,0,0,0,0,0,0,0,0,0
2,KwxdbiseRlIRNzpgvyjY0Q,axbaerf2Fk92OB4b9_peVA,e0AYjKfSF0DL-5C1CpOq6Q,4.0,0,0,0,31.0,6.0,2.0,...,0,0,0,0,0,0,0,0,0,0
3,3mwoBcTy-2gMh0L91uaIeA,_GOiybb0rImYKJfwyxEaGg,vF-uptiQ34pVLHJKzPHUlA,5.0,0,0,0,36.0,9.0,9.0,...,0,0,0,0,0,0,0,0,0,0
4,XfWf7XsBWs3kYyYq7Ns1ZQ,ojWKg3B5pH3ncAsxun3kUw,X28XK71RuEXPapeyUOwNzg,5.0,10,4,7,189.0,28.0,133.0,...,0,1,0,0,0,0,0,0,0,0


In [ ]:
train.shape

(967784, 45)

In [ ]:
train.columns

Index(['review_id', 'user_id', 'business_id', 'target', 'useful', 'funny',
       'cool', 'useful_user', 'funny_user', 'cool_user', 'date', 'year',
       'month', 'day', 'stars_business', 'review_count_business',
       'review_count', 'fans', 'average_stars', 'elite_count', 'friend_count',
       'days_yelping', 'total_compliments', 'is_open', 'postal_code',
       'cat_Restaurants', 'cat_Food', 'cat_Shopping', 'cat_Beauty_&_Spas',
       'cat_Home_Services', 'cat_Nightlife', 'cat_Health_&_Medical',
       'cat_Local_Services', 'cat_Bars', 'cat_Automotive',
       'cat_Event_Planning_&_Ser', 'cat_American_(Traditiona',
       'cat_Sandwiches', 'cat_Active_Life', 'cat_Pizza', 'cat_Coffee_&_Tea',
       'cat_Fast_Food', 'cat_American_(New)', 'cat_Breakfast_&_Brunch',
       'cat_Hotels_&_Travel'],
      dtype='object')

In [ ]:
train.rename(columns={'target': 'label'}, inplace=True)

In [ ]:
train_data, test_data = train_test_split(train, test_size=0.1, random_state=42, stratify=train["label"])

In [ ]:
# check for gpu
import torch
print(torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

True
Using device: cuda


## Functions

In [ ]:
def predictor_nn(model, X_test, rounded=True):
    model.eval()
    with torch.no_grad():
        inputs = torch.tensor(X_test.values, dtype=torch.float32).to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
    if rounded:
        predicted = torch.round(predicted.float(), decimals=0)
    return pd.DataFrame({'review_id': X_test.review_id, 'stars': predicted.cpu().numpy()})

def save_predictions(predictions, filename='predictions.csv'):
    folder = 'predictions'
    predictions.to_csv(f'{folder}/{filename}', index=False)

## Models

### AutoGluon baseline

In [ ]:
from autogluon.tabular import TabularDataset, TabularPredictor

#### 3 features left

In [ ]:
# train with gpu
predictor = TabularPredictor(label="label", eval_metric="mae", problem_type="regression").fit(train_data, presets="medium_quality", ag_args_fit={'num_gpus': 1})

No path specified. Models will be saved in: "AutogluonModels/ag-20260410_075357"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP PREEMPT_DYNAMIC Thu Jun  5 18:30:46 UTC 2025
CPU Count:          12
Pytorch Version:    2.9.1+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 8.00/8.00 GB
Total GPU Memory:   Free: 8.00 GB, Allocated: 0.00 GB, Total: 8.00 GB
GPU Count:          1
Memory Avail:       8.16 GB / 11.55 GB (70.6%)
Disk Space Avail:   774.53 GB / 1006.85 GB (76.9%)
Presets specified: ['medium_quality']
Using hyperparameters preset: hyperparameters='default'
	Consider setting `time_limit` to ensure training finishes within an expected duration or experiment with a small portion of `train_data` to identify an ideal `presets` and `hyperparameters` configuration.
Beginning AutoGluon training ...
AutoGluon wil

[1000]	valid_set's l1: 0.790289
[2000]	valid_set's l1: 0.788674
[3000]	valid_set's l1: 0.788506
[4000]	valid_set's l1: 0.788203
[5000]	valid_set's l1: 0.788062
[6000]	valid_set's l1: 0.787948
[7000]	valid_set's l1: 0.787712
[8000]	valid_set's l1: 0.787691
[9000]	valid_set's l1: 0.787471
[10000]	valid_set's l1: 0.787474


	-0.7873	 = Validation score   (-mean_absolute_error)
	132.06s	 = Training   runtime
	1.48s	 = Validation runtime
Fitting model: LightGBM ...
	Fitting with cpus=6, gpus=1, mem=0.9/7.6 GB


[1000]	valid_set's l1: 0.783955
[2000]	valid_set's l1: 0.782787


	-0.7825	 = Validation score   (-mean_absolute_error)
	35.54s	 = Training   runtime
	0.16s	 = Validation runtime
Fitting model: RandomForestMSE ...
	To force training the model, specify the model hyperparameter "ag.max_memory_usage_ratio" to a larger value (currently 1.0, set to >=1.11 to avoid the error)
		To set the same value for all models, do the following when calling predictor.fit: `predictor.fit(..., ag_args_fit={"ag.max_memory_usage_ratio": VALUE})`
		Setting "ag.max_memory_usage_ratio" to values above 1 may result in out-of-memory errors. You may consider using a machine with more memory as a safer alternative.
	Not enough memory to train RandomForestMSE... Skipping this model.
Fitting model: CatBoost ...
	Fitting with cpus=6, gpus=1, mem=1.3/7.6 GB
	Training CatBoost with GPU, note that this may negatively impact model quality compared to CPU training.
Default metric period is 5 because MAE is/are not implemented for GPU
	-0.7558	 = Validation score   (-mean_absolute_error)


[1000]	valid_set's l1: 0.784051
[2000]	valid_set's l1: 0.783827


	-0.7836	 = Validation score   (-mean_absolute_error)
	43.3s	 = Training   runtime
	0.21s	 = Validation runtime
Fitting model: WeightedEnsemble_L2 ...
	Fitting 1 model on all data | Fitting with cpus=12, gpus=0, mem=0.0/6.8 GB
	Ensemble Weights: {'NeuralNetTorch': 1.0}
	-0.694	 = Validation score   (-mean_absolute_error)
	0.05s	 = Training   runtime
	0.0s	 = Validation runtime
AutoGluon training complete, total runtime = 2418.86s ... Best model: WeightedEnsemble_L2 | Estimated inference throughput: 200899.5 rows/s (8711 batch size)
TabularPredictor saved. To load, use: predictor = TabularPredictor.load("/home/eder/projects/recommender-systems/competition2/AutogluonModels/ag-20260410_075357")


In [ ]:
predictor.fit_summary()

*** Summary of fit() ***
Estimated performance of each model:
                 model  score_val          eval_metric  pred_time_val     fit_time  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
0       NeuralNetTorch  -0.694005  mean_absolute_error       0.042363  1452.958750                0.042363        1452.958750            1       True          6
1  WeightedEnsemble_L2  -0.694005  mean_absolute_error       0.043360  1453.006358                0.000997           0.047609            2       True          8
2             CatBoost  -0.755844  mean_absolute_error       0.003300     6.697860                0.003300           6.697860            1       True          3
3             LightGBM  -0.782547  mean_absolute_error       0.163131    35.535436                0.163131          35.535436            1       True          2
4              XGBoost  -0.783190  mean_absolute_error       0.022417     8.805835                0.022417           8.805835        

/home/eder/miniconda3/envs/master312/lib/python3.12/site-packages/autogluon/core/utils/plots.py:169: UserWarning: AutoGluon summary plots cannot be created because bokeh is not installed. To see plots, please do: "pip install bokeh==2.0.1"
  warnings.warn('AutoGluon summary plots cannot be created because bokeh is not installed. To see plots, please do: "pip install bokeh==2.0.1"')


{'model_types': {'LightGBMXT': 'LGBModel',
  'LightGBM': 'LGBModel',
  'CatBoost': 'CatBoostModel',
  'NeuralNetFastAI': 'NNFastAiTabularModel',
  'XGBoost': 'XGBoostModel',
  'NeuralNetTorch': 'TabularNeuralNetTorchModel',
  'LightGBMLarge': 'LGBModel',
  'WeightedEnsemble_L2': 'WeightedEnsembleModel'},
 'model_performance': {'LightGBMXT': -0.7873389002527166,
  'LightGBM': -0.7825467686622483,
  'CatBoost': -0.7558439205782433,
  'NeuralNetFastAI': -0.7865110058322482,
  'XGBoost': -0.7831904738987508,
  'NeuralNetTorch': -0.6940052676058543,
  'LightGBMLarge': -0.7835668659609805,
  'WeightedEnsemble_L2': -0.6940052676058543},
 'model_best': 'WeightedEnsemble_L2',
 'model_paths': {'LightGBMXT': ['LightGBMXT'],
  'LightGBM': ['LightGBM'],
  'CatBoost': ['CatBoost'],
  'NeuralNetFastAI': ['NeuralNetFastAI'],
  'XGBoost': ['XGBoost'],
  'NeuralNetTorch': ['NeuralNetTorch'],
  'LightGBMLarge': ['LightGBMLarge'],
  'WeightedEnsemble_L2': ['WeightedEnsemble_L2']},
 'model_fit_times': {'Li

In [ ]:
predictor.evaluate_predictions(y_true=test_data["label"], y_pred=predictor.predict(test_data))

{'mean_absolute_error': -0.6933011567995778,
 'root_mean_squared_error': np.float64(-1.1748247019245306),
 'mean_squared_error': -1.3802130802520622,
 'r2': 0.3694331542375787,
 'pearsonr': 0.6551795811306441,
 'median_absolute_error': -0.0070421695709228516}

#### Good

In [ ]:
# Esta versión tiene las columnas de cool, funny y useful de train que no se habian incorporado por error
predictor = TabularPredictor(label="label", eval_metric="mae", problem_type="regression").fit(train_data, presets="medium_quality", ag_args_fit={'num_gpus': 1})

No path specified. Models will be saved in: "AutogluonModels/ag-20260410_095708"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP PREEMPT_DYNAMIC Thu Jun  5 18:30:46 UTC 2025
CPU Count:          12
Pytorch Version:    2.9.1+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 8.00/8.00 GB
Total GPU Memory:   Free: 8.00 GB, Allocated: 0.00 GB, Total: 8.00 GB
GPU Count:          1
Memory Avail:       7.94 GB / 11.55 GB (68.7%)
Disk Space Avail:   774.25 GB / 1006.85 GB (76.9%)
Presets specified: ['medium_quality']
Using hyperparameters preset: hyperparameters='default'
	Consider setting `time_limit` to ensure training finishes within an expected duration or experiment with a small portion of `train_data` to identify an ideal `presets` and `hyperparameters` configuration.
Beginning AutoGluon training ...
AutoGluon wil

[1000]	valid_set's l1: 0.748102
[2000]	valid_set's l1: 0.743545
[3000]	valid_set's l1: 0.74172
[4000]	valid_set's l1: 0.740662
[5000]	valid_set's l1: 0.739789
[6000]	valid_set's l1: 0.739741
[7000]	valid_set's l1: 0.73939
[8000]	valid_set's l1: 0.738811
[9000]	valid_set's l1: 0.738719
[10000]	valid_set's l1: 0.738313


	-0.7383	 = Validation score   (-mean_absolute_error)
	182.52s	 = Training   runtime
	2.53s	 = Validation runtime
Fitting model: LightGBM ...
	Fitting with cpus=6, gpus=1, mem=0.9/7.0 GB


[1000]	valid_set's l1: 0.746783


	-0.7457	 = Validation score   (-mean_absolute_error)
	72.8s	 = Training   runtime
	0.59s	 = Validation runtime
Fitting model: RandomForestMSE ...
	To force training the model, specify the model hyperparameter "ag.max_memory_usage_ratio" to a larger value (currently 1.0, set to >=1.19 to avoid the error)
		To set the same value for all models, do the following when calling predictor.fit: `predictor.fit(..., ag_args_fit={"ag.max_memory_usage_ratio": VALUE})`
		Setting "ag.max_memory_usage_ratio" to values above 1 may result in out-of-memory errors. You may consider using a machine with more memory as a safer alternative.
	Not enough memory to train RandomForestMSE... Skipping this model.
Fitting model: CatBoost ...
	Fitting with cpus=6, gpus=1, mem=1.2/7.0 GB
	Training CatBoost with GPU, note that this may negatively impact model quality compared to CPU training.
Default metric period is 5 because MAE is/are not implemented for GPU
	-0.7243	 = Validation score   (-mean_absolute_error)
	

In [ ]:
predictor.fit_summary()

*** Summary of fit() ***
Estimated performance of each model:
                 model  score_val          eval_metric  pred_time_val     fit_time  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
0       NeuralNetTorch  -0.627015  mean_absolute_error       0.118394  5560.269187                0.118394        5560.269187            1       True          6
1  WeightedEnsemble_L2  -0.627015  mean_absolute_error       0.119473  5560.316182                0.001079           0.046995            2       True          8
2             CatBoost  -0.724344  mean_absolute_error       0.113650    22.064477                0.113650          22.064477            1       True          3
3              XGBoost  -0.731059  mean_absolute_error       0.087644    29.573171                0.087644          29.573171            1       True          5
4      NeuralNetFastAI  -0.737494  mean_absolute_error       0.118776   619.346309                0.118776         619.346309        

/home/eder/miniconda3/envs/master312/lib/python3.12/site-packages/autogluon/core/utils/plots.py:169: UserWarning: AutoGluon summary plots cannot be created because bokeh is not installed. To see plots, please do: "pip install bokeh==2.0.1"
  warnings.warn('AutoGluon summary plots cannot be created because bokeh is not installed. To see plots, please do: "pip install bokeh==2.0.1"')


{'model_types': {'LightGBMXT': 'LGBModel',
  'LightGBM': 'LGBModel',
  'CatBoost': 'CatBoostModel',
  'NeuralNetFastAI': 'NNFastAiTabularModel',
  'XGBoost': 'XGBoostModel',
  'NeuralNetTorch': 'TabularNeuralNetTorchModel',
  'LightGBMLarge': 'LGBModel',
  'WeightedEnsemble_L2': 'WeightedEnsembleModel'},
 'model_performance': {'LightGBMXT': -0.738300207138041,
  'LightGBM': -0.7456960294319991,
  'CatBoost': -0.7243443036952504,
  'NeuralNetFastAI': -0.7374935798879853,
  'XGBoost': -0.7310589809886296,
  'NeuralNetTorch': -0.6270152386369859,
  'LightGBMLarge': -0.7375958239194166,
  'WeightedEnsemble_L2': -0.6270152386369859},
 'model_best': 'WeightedEnsemble_L2',
 'model_paths': {'LightGBMXT': ['LightGBMXT'],
  'LightGBM': ['LightGBM'],
  'CatBoost': ['CatBoost'],
  'NeuralNetFastAI': ['NeuralNetFastAI'],
  'XGBoost': ['XGBoost'],
  'NeuralNetTorch': ['NeuralNetTorch'],
  'LightGBMLarge': ['LightGBMLarge'],
  'WeightedEnsemble_L2': ['WeightedEnsemble_L2']},
 'model_fit_times': {'Lig

In [ ]:
predictor.evaluate_predictions(y_true=test_data["label"], y_pred=predictor.predict(test_data))

{'mean_absolute_error': -0.6394110887361699,
 'root_mean_squared_error': np.float64(-1.1186028674510708),
 'mean_squared_error': -1.2512723750697579,
 'r2': 0.42834125684905333,
 'pearsonr': 0.6957110105315124,
 'median_absolute_error': -0.0034494400024414062}

In [ ]:
test = pd.read_csv('data/test_final.csv')
preds = predictor.predict(test)
preds = pd.DataFrame({'review_id': test.review_id, 'stars': preds})
save_predictions(preds, 'AutoGluonNeuralNetTorch.csv')

In [ ]:
preds.stars = round(preds.stars, 0)
save_predictions(preds, 'AutoGluonNeuralNetTorchRounded.csv')

In [ ]:
test

In [5]:
test = pd.read_csv('data/test_final.csv')
predictor = TabularPredictor.load('AutogluonModels/ag-20260410_095708')
preds = predictor.predict(test)
preds = pd.DataFrame({'review_id': test.review_id, 'stars': preds})
preds.stars = round(preds.stars, 0)
save_predictions(preds, 'AutoGluonNeuralNetTorchRoundedGood.csv')


In [6]:
res1 = pd.read_csv('predictions/AutoGluonNeuralNetTorchRoundedGood.csv')
res2 = pd.read_csv('predictions/AutoGluonNeuralNetTorchRounded.csv')

In [7]:
# Calculate the mae of column "stars" in res1 and res2
from sklearn.metrics import mean_absolute_error
mae = mean_absolute_error(res1["stars"], res2["stars"])
print(f"MAE: {mae}")

MAE: 0.22338191506033536


In [8]:
# calculate the accumulated distance of each prediction in res1 and res2
accumulated_distance = np.sum(np.abs(res1["stars"] - res2["stars"]))
print(f"Accumulated distance: {accumulated_distance}")

Accumulated distance: 92651.0


#### One hot encoded data

In [4]:
train = pd.read_csv('data/train_final_ohe.csv')
train.shape

(967784, 45)

In [ ]:
predictor = TabularPredictor(label="label", eval_metric="mae", problem_type="regression").fit(train_data, presets="medium_quality", ag_args_fit={'num_gpus': 1})

### DeepFM

In [ ]:
from models import RecVAE
train_df = pd.read_csv("train_reviews.csv")

recommender = RecVAE(use_gpu=True, verbose=True)
recommender.fit(train_df)

print(recommender.predict(user_id="Ha3iJu77CxlrFm-vQRs_8g",
                          business_id="tnhfDv5Il8EaGSXZGiuQGg"))

print(recommender.recommend(user_id="Ha3iJu77CxlrFm-vQRs_8g", k=5))

## V2
### PREPROCESAMIENTO
Esta versión hereda todo el motor de preprocesamiento de características de la V1 (aplanamiento de atributos y tratamiento híbrido de categorías), pero abandona el split aleatorio para adoptar un marco de evaluación del mundo real, incorporando la identidad de los actores clave.

- Split Temporal Estricto: Partición cronológica basada en date_num (70% Train, 15% Val, 15% Test). Esto garantiza que el modelo aprenda del pasado para predecir el futuro, eliminando el sesgo de "viaje en el tiempo" (Data Leakage) presente en la V1.

- Inclusión de Identidades (user_id y business_id): A diferencia de la V1 (donde se descartaban), en esta versión se agregan explícitamente como variables de entrenamiento. Esto permite que el modelo genere embeddings de identidad, reconociendo y capturando los patrones de comportamiento de usuarios y negocios recurrentes.

- Limpieza de Control Temporal: Se elimina de la memoria únicamente la columna técnica date_num tras realizar la partición cronológica, evitando que la red neuronal use el timestamp numérico como un atajo o regla de decisión simplista.

In [7]:
import preprocess_autogluon2
import pandas as pd
# %run preprocess_autogluon2.py
train = pd.read_parquet('data/train_ag.parquet')
train

,target,user_id,business_id,date_num,review_useful,review_funny,review_cool,date,review_year,review_month,...,attr_RestaurantsCounterService,attr_BusinessParking,attr_Music,attr_DietaryRestrictions_dairy-free,attr_DietaryRestrictions_gluten-free,attr_DietaryRestrictions_vegan,attr_DietaryRestrictions_kosher,attr_DietaryRestrictions_halal,attr_DietaryRestrictions_soy-free,attr_DietaryRestrictions_vegetarian
0,5.0,-1BSu2dt_rOAqllw9ZDXtA,smkZq4G1AOm4V6p3id5sww,1475250572,0,0,0,2016-09-30 15:49:32,2016,9,...,NaN,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1.0,xpe178ng_gj5X6HgqtOing,96_c_7twb7hYRZ9HHrq01g,1607524791,2,0,1,2020-12-09 14:39:51,2020,12,...,NaN,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4.0,axbaerf2Fk92OB4b9_peVA,e0AYjKfSF0DL-5C1CpOq6Q,1378311591,0,0,0,2013-09-04 16:19:51,2013,9,...,NaN,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,5.0,_GOiybb0rImYKJfwyxEaGg,vF-uptiQ34pVLHJKzPHUlA,1551529454,0,0,0,2019-03-02 12:24:14,2019,3,...,NaN,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5.0,ojWKg3B5pH3ncAsxun3kUw,X28XK71RuEXPapeyUOwNzg,1587666389,10,4,7,2020-04-23 18:26:29,2020,4,...,NaN,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
967779,5.0,iZAAjkPZ0sopzfajcfdOUg,2PvPsZ3KRFCtHbQkNHvGpg,1587310932,1,0,0,2020-04-19 15:42:12,2020,4,...,NaN,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN
967780,4.0,AN8XscFSH1jctLSqAQ9-bA,-K0zTgGyxo-AeSkcV0IVaA,1392056823,0,0,0,2014-02-10 18:27:03,2014,2,...,NaN,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN
967781,5.0,qtgODIPIsKsH2j3rItF9Tw,eZCt3doDaA-l4sg_OM67YQ,1563977992,0,0,1,2019-07-24 14:19:52,2019,7,...,NaN,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN
967782,5.0,gu7nU1IM7U3lLwUbqLeiDQ,ww3YJXu5c18aGZXWmm00qg,1320734285,0,0,0,2011-11-08 06:38:05,2011,11,...,NaN,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### ENTRENAMIENTO

In [8]:
import train_autogluon2
%run train_autogluon2.py

c:\Users\kzzazzk\OneDrive\Documentos\MAADM\2º Cuatrimestre\2S\RECSYS\recommender-systems\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



█████████████████████████████████████████████████████████████████
  AUTOGLUON — SISTEMA DE RECOMENDACIÓN YELP (v2)
█████████████████████████████████████████████████████████████████

  Cargando datos y haciendo splits temporales...

  Splits por fecha (date_num ascendente):
    Train: [0:677448] = 677,448 reviews (70%)
    Val:   [677448:822616] = 145,168 reviews (15%)
    Test:  [822616:967784] = 145,168 reviews (15%)
  Sanitizando para AutoGluon...


Preset alias specified: 'high_v150' maps to 'high_quality_v150'.
Verbosity: 2 (Standard Logging)



  Estadísticas Target:
    Train — min:1  max:5  mean:3.764  std:1.432
    Val   — min:1  max:5  mean:3.782  std:1.541

  Columnas TEXT (NLP): ['categories']
  Columnas categóricas (inferidas): []

  Iniciando entrenamiento (time_limit=172800s = 48.0h)...


=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.10
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          16
Pytorch Version:    2.11.0+cu130
CUDA Version:       13.0
GPU Memory:         GPU 0: 8.00/8.00 GB
Total GPU Memory:   Free: 8.00 GB, Allocated: 0.00 GB, Total: 8.00 GB
GPU Count:          1
Memory Avail:       13.72 GB / 31.90 GB (43.0%)
Disk Space Avail:   191.90 GB / 930.64 GB (20.6%)
Presets specified: ['high_v150']
Stack configuration (auto_stack=True): num_stack_levels=0, num_bag_folds=0, num_bag_sets=1
Beginning AutoGluon training ... Time limit = 172800s
AutoGluon will save models to "c:\Users\kzzazzk\OneDrive\Documentos\MAADM\2º Cuatrimestre\2S\RECSYS\recommender-systems\competition2\models2\autogluon_1776010975"
Train Data Rows:    677448
Train Data Columns: 138
Tuning Data Rows:    145168
Tuning Data Columns: 138
Label Column:       target
Problem Type:       regres


  ✓ Entrenamiento completado en 38.2 min

═════════════════════════════════════════════════════════════════
  LEADERBOARD
═════════════════════════════════════════════════════════════════
                            model  score_val     fit_time  pred_time_val
0             WeightedEnsemble_L2  -0.632458  1070.490630       5.018710
1       NeuralNetTorchRegularized  -0.632537   280.219301       2.562197
2          NeuralNetTorchStandard  -0.636978   790.185908       2.454000
3        WeightedEnsemble_L2_FULL        NaN  1167.634982            NaN
4     NeuralNetTorchStandard_FULL        NaN   877.859970            NaN
5  NeuralNetTorchRegularized_FULL        NaN   289.689591            NaN

═════════════════════════════════════════════════════════════════
  FEATURE IMPORTANCE — top-30 (puede tardar ~2 min)
═════════════════════════════════════════════════════════════════


	196.19s	= Expected runtime (65.4s per shuffle set)
	112.37s	= Actual runtime (Completed 3 of 3 shuffle sets)


                             importance    stddev   p_value  n  p99_high   p99_low
average_stars                  0.596538  0.039080  0.000714  3  0.820473  0.372603
stars_business                 0.241340  0.010145  0.000294  3  0.299474  0.183207
review_useful                  0.100601  0.009310  0.001421  3  0.153950  0.047252
review_cool                    0.092628  0.011436  0.002521  3  0.158156  0.027100
review_funny                   0.024886  0.003131  0.002617  3  0.042824  0.006948
review_count                   0.018233  0.002033  0.002059  3  0.029881  0.006585
useful                         0.013759  0.004356  0.015910  3  0.038717 -0.011200
useful_per_review              0.011487  0.002347  0.006814  3  0.024934 -0.001960
cool                           0.005144  0.001974  0.022861  3  0.016453 -0.006164
review_count_business          0.003864  0.001110  0.013212  3  0.010224 -0.002496
funny                          0.002227  0.001700  0.075630  3  0.011967 -0.007512
city

### INFERENCIA

In [9]:
import pandas as pd
import os
from autogluon.tabular import TabularPredictor, TabularDataset

# ─────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────
MODEL_PATH = "models2/autogluon_1776010975"   # ⚠️ cambia si usaste timestamp
TEST_PATH  = "data/test_ag.parquet"
RAW_TEST_PATH = "data/test_reviews.csv"      # ⚠️ Ajusta la ruta a tu test_reviews.csv original
OUTPUT_PATH = "submissions/submission_v2.csv"

TARGET_COL = "target"
TEXT_FEATURES = ["categories"]


# ─────────────────────────────────────────────────────────────
# SANITIZE (MISMA QUE TRAIN)
# ─────────────────────────────────────────────────────────────
def sanitize_for_autogluon(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Nullable ints → float32
    nullable_int_cols = [
        c for c in df.columns
        if str(df[c].dtype).startswith("Int")
    ]
    for c in nullable_int_cols:
        df[c] = df[c].astype("float32")

    # TEXT features
    for c in TEXT_FEATURES:
        if c in df.columns:
            df[c] = df[c].astype("string")

    # Object → category
    obj_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()
    for c in obj_cols:
        if c not in TEXT_FEATURES:
            df[c] = df[c].astype("category")

    return df


# ─────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────
def main():
    print("\n🚀 Cargando modelo...")
    predictor = TabularPredictor.load(MODEL_PATH)

    print("📂 Cargando test data...")
    test_data = TabularDataset(TEST_PATH)
    test_data = sanitize_for_autogluon(test_data)

    # Cargar solo los review_id originales para no saturar la RAM
    print("📂 Cargando review_ids originales...")
    original_test_ids = pd.read_csv(RAW_TEST_PATH, usecols=["review_id"])

    # Validar que tengan la misma cantidad de filas
    if len(original_test_ids) != len(test_data):
        print(f"⚠️ ADVERTENCIA: Las filas en {RAW_TEST_PATH} ({len(original_test_ids)}) "
              f"no coinciden con {TEST_PATH} ({len(test_data)})")

    print("🤖 Generando predicciones...")
    preds = predictor.predict(test_data, model=predictor.model_best)

    # Clipping (Yelp: 1–5 estrellas)
    preds = preds.clip(1, 5)

    print("💾 Guardando submission...")
    os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

    # Asignar el review_id real y los valores predichos
    submission = pd.DataFrame({
        "review_id": original_test_ids["review_id"],
        "stars": preds.values # .values asegura que se asigne correctamente ignorando índices de pandas
    })
    
    submission.to_csv(OUTPUT_PATH, index=False)

    print("\n✅ DONE")
    print(f"📄 Archivo: {OUTPUT_PATH}")
    print(f"📊 Stats → min={preds.min():.3f} max={preds.max():.3f} "
          f"mean={preds.mean():.3f} std={preds.std():.3f}")


if __name__ == "__main__":
    main()


🚀 Cargando modelo...
📂 Cargando test data...


Loaded data from: data/test_ag.parquet | Columns = 139 / 139 | Rows = 414765 -> 414765


📂 Cargando review_ids originales...
🤖 Generando predicciones...
💾 Guardando submission...

✅ DONE
📄 Archivo: submissions/submission_v2.csv
📊 Stats → min=1.000 max=5.000 mean=3.896 std=1.365


Los resultados de la inferencia dan un MAE en test de 0.6519 el cual empeora ligeramente el conseguido con v1 de 0.6393

## V3
### PREPROCESAMIENTO
Mientras que la V2 introdujo la partición temporal para evaluar el modelo correctamente, la V3 lleva el concepto de "tiempo" al interior de las features, convirtiendo datos estáticos en métricas dinámicas que evolucionan junto con el usuario.

- Ingeniería de Características en Ventana Expandible (Expanding Windows): Se reemplazan los promedios globales estáticos por promedios "hasta la fecha".

    - user_avg_stars_at_time / biz_avg_stars_at_time: El modelo ya no ve la nota media total de un usuario o negocio, sino la nota media exacta que tenían en el instante anterior a la reseña actual.

    - user_review_count_at_time / biz_review_count_at_time: Refleja la cantidad de reseñas acumuladas hasta ese momento exacto, indicando la fiabilidad del promedio.

    - delta_stars: Se introduce una métrica relacional que calcula la diferencia directa entre la exigencia histórica del usuario y la calidad histórica del negocio al momento de la interacción.

- Detección de Cold Start (Arranque en Frío): * Creación de flags explícitos (is_cold_user, is_cold_biz) para señalar a la red neuronal cuándo un usuario o negocio no tiene historial previo. Esto permite que el modelo active estrategias diferentes (por ejemplo, basarse exclusivamente en el perfil de precios o categorías del local) cuando las medias históricas son nulas.

- Contextualización del Engagement:

    - user_seniority_days: Se calcula la antigüedad del usuario en el momento exacto de escribir la reseña, reemplazando a la métrica estática y ruidosa de days_yelping.

    - was_elite_at_review: En lugar de usar un conteo total de años como "Elite", se verifica si el usuario ostentaba ese estatus específico en el año en que se publicó la valoración.

- Eliminación Definitiva de Variables Contaminadas: Se descartan proactivamente todas las columnas que contenían información del futuro, agregaciones estáticas globales o métricas post-reseña (average_stars, stars_business, review_count, review_count_business, review_cool, review_funny, review_useful, years_yelping), garantizando un entorno de predicción 100% estricto y libre de fugas de datos (Data Leakage).

In [10]:
import preprocess_autogluon3
import pandas as pd
# %run preprocess_autogluon3.py
train = pd.read_parquet('data2/train_ag.parquet')
train

⚠ AVISO: data2\test_final.csv no existe. Usando fallback: data2\test_reviews.csv


,target,user_id,business_id,review_year,review_month,review_dow,review_is_weekend,useful,funny,cool,...,attr_DietaryRestrictions_vegetarian,user_seniority_days,was_elite_at_review,user_avg_stars_at_time,user_review_count_at_time,biz_avg_stars_at_time,biz_review_count_at_time,is_cold_user,is_cold_biz,delta_stars
0,3.0,3MYdpmHeNwC6FquRWi3YOg,8fWtmzdexsuVnxuZx_vPVw,2005,3,1,0,236.0,252.0,114.0,...,NaN,10,0,3.759882,0,3.759882,0,1,1,0.00
1,4.0,58yhbFfNHjULDZx0FD-Dvw,50yKOLjTxUXKkLVYrAHRAw,2005,3,2,0,55.0,43.0,20.0,...,NaN,0,0,3.759882,1,3.759882,1,1,1,0.00
2,4.0,58yhbFfNHjULDZx0FD-Dvw,xwKYBPO0ByGlkvNcr8FdqQ,2005,3,2,0,55.0,43.0,20.0,...,NaN,1,0,4.000000,2,3.759882,2,0,1,0.24
3,3.0,58yhbFfNHjULDZx0FD-Dvw,IKMAgK2m6WRIViVFB2vAFQ,2005,3,2,0,55.0,43.0,20.0,...,NaN,1,0,4.000000,3,3.759882,3,0,1,0.24
4,4.0,3zBJUlWtPNoZ0uN83ODbyg,0uxg3_noVCE78Wgjb8DSyA,2005,3,0,0,176.0,111.0,80.0,...,NaN,56,0,3.759882,4,3.759882,4,1,1,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
967779,5.0,909VYxYlC3ZZMaeDcZWmzw,u50hTvPV_W_Hx625ytvLYw,2022,1,2,0,0.0,0.0,0.0,...,NaN,46,0,3.759882,1,3.481482,2,1,0,0.28
967780,5.0,SfhWhnf-TuTjEulNwhVasA,b_DNC7vqQGYWw59Hntpl7Q,2022,1,2,0,0.0,0.0,0.0,...,NaN,812,0,3.759882,1,5.000000,3,1,0,-1.24
967781,5.0,MvTxxHzwrWKiloCrgTySYw,r926yelr_EJAKQCjdD6AWA,2022,1,2,0,0.0,0.0,0.0,...,NaN,990,0,3.759882,1,4.148248,4,1,0,-0.39
967782,5.0,BlvDcYBfjf55OuCLfixEsQ,Eo0psaCr_gcty9QqyUy46Q,2022,1,2,0,92.0,21.0,30.0,...,NaN,2846,0,4.000000,1,2.726531,1,0,0,1.27


### ENTRENAMIENTO

In [11]:
import train_autogluon3
%run train_autogluon3.py


█████████████████████████████████████████████████████████████████
  AUTOGLUON — SISTEMA DE RECOMENDACIÓN YELP (v3 — features temporales)
█████████████████████████████████████████████████████████████████

  Cargando datos y haciendo splits temporales...
  ⚠ Columna date_num no encontrada. No se puede hacer split temporal.
    Usando split aleatorio 70/15/15
  Sanitizando para AutoGluon...


Preset alias specified: 'high_v150' maps to 'high_quality_v150'.
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.10
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          16
Pytorch Version:    2.11.0+cu130
CUDA Version:       13.0
GPU Memory:         GPU 0: 7.98/8.00 GB
Total GPU Memory:   Free: 7.98 GB, Allocated: 0.02 GB, Total: 8.00 GB
GPU Count:          1
Memory Avail:       13.29 GB / 31.90 GB (41.7%)
Disk Space Avail:   191.07 GB / 930.64 GB (20.5%)
Presets specified: ['high_v150']
Stack configuration (auto_stack=True): num_stack_levels=0, num_bag_folds=0, num_bag_sets=1



  Estadísticas Target:
    Train — min:1  max:5  mean:3.760  std:1.480
    Val   — min:1  max:5  mean:3.760  std:1.479

  Columnas TEXT (NLP): ['categories']
  Columnas categóricas (inferidas): []

  Iniciando entrenamiento (time_limit=172800s = 48.0h)...


Beginning AutoGluon training ... Time limit = 172800s
AutoGluon will save models to "c:\Users\kzzazzk\OneDrive\Documentos\MAADM\2º Cuatrimestre\2S\RECSYS\recommender-systems\competition2\models2\autogluon3"
Train Data Rows:    677448
Train Data Columns: 137
Tuning Data Rows:    145168
Tuning Data Columns: 137
Label Column:       target
Problem Type:       regression
Preprocessing data ...
Using Feature Generators to preprocess the data ...
Fitting AutoMLPipelineFeatureGenerator...
	Available Memory:                    13393.97 MB
	Train Data (Original)  Memory Usage: 805.55 MB (6.0% of available memory)
	Stage 1 Generators:
		Fitting AsTypeFeatureGenerator...
			Note: Converting 11 features to boolean dtype as they only contain 2 unique values.
	Stage 2 Generators:
		Fitting FillNaFeatureGenerator...
	Stage 3 Generators:
		Fitting IdentityFeatureGenerator...
		Fitting CategoryFeatureGenerator...
			Fitting CategoryMemoryMinimizeFeatureGenerator...
		Fitting TextSpecialFeatureGenerator.


  ✓ Entrenamiento completado en 35.5 min

═════════════════════════════════════════════════════════════════
  LEADERBOARD
═════════════════════════════════════════════════════════════════
                            model  score_val     fit_time  pred_time_val
0             WeightedEnsemble_L2  -0.905147  1073.403452       5.169833
1       NeuralNetTorchRegularized  -0.905188   278.011218       2.603554
2          NeuralNetTorchStandard  -0.908820   795.312297       2.564273
3        WeightedEnsemble_L2_FULL        NaN  1002.062086            NaN
4     NeuralNetTorchStandard_FULL        NaN   740.516324            NaN
5  NeuralNetTorchRegularized_FULL        NaN   261.465825            NaN

═════════════════════════════════════════════════════════════════
  FEATURE IMPORTANCE — top-30 (puede tardar ~2 min)
═════════════════════════════════════════════════════════════════


	185.91s	= Expected runtime (61.97s per shuffle set)
	111.58s	= Actual runtime (Completed 3 of 3 shuffle sets)


                             importance    stddev   p_value  n  p99_high   p99_low
biz_avg_stars_at_time          0.161252  0.003573  0.000082  3  0.181724  0.140780
useful_per_review              0.117714  0.004263  0.000218  3  0.142140  0.093287
user_avg_stars_at_time         0.041458  0.001712  0.000284  3  0.051266  0.031649
useful                         0.037307  0.001222  0.000179  3  0.044311  0.030302
cool                           0.030495  0.003030  0.001637  3  0.047858  0.013132
review_year                    0.019273  0.004076  0.007292  3  0.042631 -0.004084
delta_stars                    0.018306  0.001880  0.001748  3  0.029076  0.007536
funny                          0.017737  0.002011  0.002129  3  0.029260  0.006214
friend_count                   0.006746  0.005052  0.073423  3  0.035696 -0.022203
fans                           0.006674  0.001326  0.006455  3  0.014274 -0.000926
is_cold_user                   0.006441  0.001663  0.010755  3  0.015971 -0.003089
user

AttributeError: 'TabularPredictor' object has no attribute 'get_model_best'

### INFERENCIA Y RESULTADOS

In [12]:
import pandas as pd
import os
from autogluon.tabular import TabularPredictor, TabularDataset

# ─────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────
MODEL_PATH = "models2/autogluon3"   # ⚠️ cambia si usaste timestamp
TEST_PATH  = "data2/test_ag.parquet"
RAW_TEST_PATH = "data2/test_reviews.csv"      # ⚠️ Ajusta la ruta a tu test_reviews.csv original
OUTPUT_PATH = "submissions/submission_v3.csv"

TARGET_COL = "target"
TEXT_FEATURES = ["categories"]


# ─────────────────────────────────────────────────────────────
# SANITIZE (MISMA QUE TRAIN)
# ─────────────────────────────────────────────────────────────
def sanitize_for_autogluon(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Nullable ints → float32
    nullable_int_cols = [
        c for c in df.columns
        if str(df[c].dtype).startswith("Int")
    ]
    for c in nullable_int_cols:
        df[c] = df[c].astype("float32")

    # TEXT features
    for c in TEXT_FEATURES:
        if c in df.columns:
            df[c] = df[c].astype("string")

    # Object → category
    obj_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()
    for c in obj_cols:
        if c not in TEXT_FEATURES:
            df[c] = df[c].astype("category")

    return df


# ─────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────
def main():
    print("\n🚀 Cargando modelo...")
    predictor = TabularPredictor.load(MODEL_PATH)

    print("📂 Cargando test data...")
    test_data = TabularDataset(TEST_PATH)
    test_data = sanitize_for_autogluon(test_data)

    # Cargar solo los review_id originales para no saturar la RAM
    print("📂 Cargando review_ids originales...")
    original_test_ids = pd.read_csv(RAW_TEST_PATH, usecols=["review_id"])

    # Validar que tengan la misma cantidad de filas
    if len(original_test_ids) != len(test_data):
        print(f"⚠️ ADVERTENCIA: Las filas en {RAW_TEST_PATH} ({len(original_test_ids)}) "
              f"no coinciden con {TEST_PATH} ({len(test_data)})")

    print("🤖 Generando predicciones...")
    preds = predictor.predict(test_data, model=predictor.model_best)

    # Clipping (Yelp: 1–5 estrellas)
    preds = preds.clip(1, 5)

    print("💾 Guardando submission...")
    os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

    # Asignar el review_id real y los valores predichos
    submission = pd.DataFrame({
        "review_id": original_test_ids["review_id"],
        "stars": preds.values # .values asegura que se asigne correctamente ignorando índices de pandas
    })
    
    submission.to_csv(OUTPUT_PATH, index=False)

    print("\n✅ DONE")
    print(f"📄 Archivo: {OUTPUT_PATH}")
    print(f"📊 Stats → min={preds.min():.3f} max={preds.max():.3f} "
          f"mean={preds.mean():.3f} std={preds.std():.3f}")


if __name__ == "__main__":
    main()


🚀 Cargando modelo...
📂 Cargando test data...


Loaded data from: data2/test_ag.parquet | Columns = 137 / 137 | Rows = 414765 -> 414765


📂 Cargando review_ids originales...
🤖 Generando predicciones...
💾 Guardando submission...

✅ DONE
📄 Archivo: submissions/submission_v3.csv
📊 Stats → min=1.000 max=5.000 mean=4.101 std=1.093


Los resultados de la inferencia dan un MAE de test de 1.3866 el cual es el peor hasta ahora.

## COMPROBACIÓN DE CORRELACIÓN DE MALOS RESULTADOS CON EL TEMPORAL SPLIT
Entrenaremos con un split aleatorio sin tener en cuenta el tiempo para comprobar si la razón de los malos rendimientos es el temporal split o los campos eliminados/añadidos en las versiones v2 y v3

### V2 pero sin temporal split

In [18]:
"""
train_autogluon.py  (v2 — post análisis + splits temporales)
============================================================
Cambios en esta versión:
  - Carga y ordena reviews por date_num (timestamp ascendente)
  - Hace splits temporales: 70% train, 15% val, 15% test
  - NO mezcla índices entre splits (mantiene orden cronológico)
  - Usa tuning_data del split temporal como validación interna

Dataset:
  - 1.201 categorías únicas → 'categories' como TEXT (NLP + embeddings)
  - 81 atributos → muchos son BusinessParking_* y Ambience_* (binarios)
  - 15 atributos categóricos (WiFi, NoiseLevel…) → AutoGluon label-encode
  - ~968K reviews, ~30K negocios
  - num_bag_folds=5 y num_stack_levels=1 configurados para tiempo viable

Parámetros importantes:
  - TIME_LIMIT = 14400 * 12 → 48 horas (para GPU con mucha RAM)
    Ajusta según hardware: GPU → 14400 (4h), CPU solo → 7200 (2h)
"""

import pandas as pd
import numpy as np
import os
import gc
import time

from autogluon.tabular import TabularPredictor, TabularDataset
from autogluon.common.features.feature_metadata import FeatureMetadata


# ─────────────────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────────────────
DATA_DIR   = "data"
MODELS_DIR = "models2/autogluon_1776010975_no_temporal_split"

TRAIN_PATH  = os.path.join(DATA_DIR, "train_ag.parquet")
TEST_PATH   = os.path.join(DATA_DIR, "test_ag.parquet")

TARGET_COL  = "target"
EVAL_METRIC = "mae"

# Ajustar según hardware:
#   GPU + mucha RAM → 14400 (4h)
#   Solo CPU        → 7200  (2h)
TIME_LIMIT = 14400 * 12 # -> 48 horas
SEED       = 42


# ─────────────────────────────────────────────────────────────────────────────
# FEATURE METADATA
# Forzamos 'categories' como TEXT para activar los módulos NLP.
# El resto AutoGluon lo infiere bien a partir de los dtypes del parquet.
# ─────────────────────────────────────────────────────────────────────────────

# Columnas que deben tratarse como TEXT (NLP)
# → Solo 'categories' (string largo, multi-valor separado por comas)
TEXT_FEATURES = ["categories"]

# Las siguientes las infiere AutoGluon como 'category' a partir de dtype object:
# top_category, city, state, postal_code, elite_bucket,
# attr_WiFi, attr_NoiseLevel, attr_RestaurantsAttire, attr_Alcohol, attr_Smoking…


# ─────────────────────────────────────────────────────────────────────────────
# HIPERPARÁMETROS
# ─────────────────────────────────────────────────────────────────────────────

HYPERPARAMETERS = {
    "NN_TORCH": [
        {
            "num_epochs": 50,
            "learning_rate": 1e-3,
            "dropout_prob": 0.1,
            "weight_decay": 1e-6,
            "batch_size": 256,  # <-- ¡Bajamos a 256!
            "ag_args": {"name_suffix": "Standard"},
        },
        {
            "num_epochs": 30,
            "learning_rate": 3e-4,
            "dropout_prob": 0.3,
            "weight_decay": 1e-5,
            "batch_size": 512, # <-- ¡Bajamos a 512!
            "ag_args": {"name_suffix": "Regularized"},
        },
    ],
}

from autogluon.common.features.feature_metadata import FeatureMetadata

TEXT_FEATURES = ["categories"]

from sklearn.model_selection import train_test_split

def load_and_split_baseline(parquet_path: str):
    """
    Carga el parquet y hace exactamente el mismo split 90/10 de la baseline,
    ignorando el tiempo.
    """
    df = pd.read_parquet(parquet_path)
    
    # Quitamos la columna técnica de tiempo para que el modelo no la vea
    cols_to_drop = ["date_num"]
    df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True, errors="ignore")
    
    print("\n  [⚠️] Ejecutando Split Aleatorio Baseline (90% Train / 10% Test)...")
    
    # Redondeamos el target temporalmente para que stratify funcione sin fallos
    stratify_col = df[TARGET_COL].round().astype(int)
    
    train_data, test_data = train_test_split(
        df, 
        test_size=0.1, 
        random_state=42, 
        stratify=stratify_col
    )
    
    print(f"    Train: {len(train_data):,} reviews")
    print(f"    Test:  {len(test_data):,} reviews")
    
    return train_data.reset_index(drop=True), test_data.reset_index(drop=True)

def sanitize_for_autogluon(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # 1) Nullable integers de pandas -> float32 para evitar warnings de numpy/AutoGluon
    nullable_int_cols = [
        c for c in df.columns
        if str(df[c].dtype).startswith("Int")
    ]
    for c in nullable_int_cols:
        df[c] = df[c].astype("float32")

    # 2) Mantener el texto como texto
    for c in TEXT_FEATURES:
        if c in df.columns:
            df[c] = df[c].astype("string")

    # 3) El resto de object/string -> category para evitar detección errónea de datetime
    obj_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()
    for c in obj_cols:
        if c not in TEXT_FEATURES:
            df[c] = df[c].astype("category")

    return df


# ─────────────────────────────────────────────────────────────────────────────
# ENTRENAMIENTO
# ─────────────────────────────────────────────────────────────────────────────

def train(time_limit: int = TIME_LIMIT):
    print("\n  Cargando datos y haciendo split aleatorio...")
    
    # 1. Llamamos a la nueva función
    train_data, test_data = load_and_split_baseline(TRAIN_PATH)
    
    # 2. Aplicar sanitización
    print("  Sanitizando para AutoGluon...")
    train_data = sanitize_for_autogluon(train_data)
    test_data = sanitize_for_autogluon(test_data)
    
    print(f"\n  Estadísticas Target Train:")
    print(f"    min:{train_data[TARGET_COL].min():.0f}  "
          f"max:{train_data[TARGET_COL].max():.0f}  "
          f"mean:{train_data[TARGET_COL].mean():.3f}  "
          f"std:{train_data[TARGET_COL].std():.3f}")

    text_cols = [c for c in TEXT_FEATURES if c in train_data.columns]
    
    predictor = TabularPredictor(
        label        = TARGET_COL,
        eval_metric  = EVAL_METRIC,
        path         = MODELS_DIR,
        problem_type = "regression",
        verbosity    = 2,
    )
    
    feature_metadata = FeatureMetadata.from_df(train_data)
    feature_metadata = feature_metadata.add_special_types(
        {col: ["text"] for col in TEXT_FEATURES if col in train_data.columns}
    )

    print(f"\n  Iniciando entrenamiento (time_limit={time_limit}s)...")
    t0 = time.time()

    predictor.fit(
        train_data   = train_data,
        # OJO AQUÍ: Hemos borrado tuning_data=val_data
        # AutoGluon cogerá un % automáticamente como en tu notebook
        hyperparameters = HYPERPARAMETERS,
        time_limit   = time_limit,
        presets      = "high_v150",
        ag_args_fit  = {"num_gpus": 1},
        num_stack_levels = 0,
        num_bag_folds    = 0,
        feature_metadata = feature_metadata,
        excluded_model_types = ["KNN"],
    )

    elapsed = time.time() - t0
    print(f"\n  ✓ Entrenamiento completado en {elapsed/60:.1f} min")

    return predictor, test_data

# ─────────────────────────────────────────────────────────────────────────────
# EVALUACIÓN
# ─────────────────────────────────────────────────────────────────────────────

def evaluate(predictor: TabularPredictor, val_data: pd.DataFrame = None):
    print("\n" + "═"*65)
    print("  LEADERBOARD")
    print("═"*65)
    lb = predictor.leaderboard(silent=True)
    print(lb[["model","score_val","fit_time","pred_time_val"]].to_string())

    print("\n" + "═"*65)
    print("  FEATURE IMPORTANCE — top-30 (puede tardar ~2 min)")
    print("═"*65)
    try:
        if val_data is not None:
            data_for_fi = val_data
        else:
            data_for_fi = TabularDataset(TRAIN_PATH)
        
        fi = predictor.feature_importance(
            data             = data_for_fi,
            num_shuffle_sets = 3,
            subsample_size   = min(5000, len(data_for_fi)),
        )
        print(fi.head(30).to_string())

        # Guardar para análisis posterior
        fi.to_csv("models2/feature_importance.csv")
        print("\n  → Guardado en models/feature_importance.csv")
    except Exception as e:
        print(f"  Feature importance no disponible: {e}")

    best = predictor.model_best

    lb = predictor.leaderboard(silent=True)
    val_mae = -lb.loc[lb["model"] == best, "score_val"].values[0]
    print(f"\n  Mejor modelo : {best}")
    print(f"  MAE (val)    : {val_mae:.4f}")


# ─────────────────────────────────────────────────────────────────────────────
# PREDICCIÓN
# ─────────────────────────────────────────────────────────────────────────────

def predict(predictor: TabularPredictor,
            test_data: pd.DataFrame = None,
            output_path: str = "submissions/submission.csv"):
    print("\n" + "═"*65)
    print("  PREDICCIÓN TEST")
    print("═"*65)

    if test_data is None:
        print("  Cargando test_data desde archivo...")
        test_data = TabularDataset(TEST_PATH)
        test_data = sanitize_for_autogluon(test_data)
    
    preds = predictor.predict(test_data, model=predictor.model_best)
    # Las estrellas de Yelp son siempre [1, 5]
    preds = preds.clip(1, 5)

    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    pd.DataFrame({"stars": preds}).to_csv(output_path, index=True, index_label="index")

    print(f"  ✓ Guardado: {output_path}")
    print(f"  min={preds.min():.3f}  max={preds.max():.3f}  "
          f"mean={preds.mean():.3f}  std={preds.std():.3f}")
    return preds


def load_predictor() -> TabularPredictor:
    print(f"  Cargando predictor desde {MODELS_DIR}...")
    return TabularPredictor.load(MODELS_DIR)


# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────
# --- EJECUCIÓN MANUAL EN NOTEBOOK ---
print("\n" + "█"*65)
print("  AUTOGLUON — SISTEMA DE RECOMENDACIÓN YELP (RANDOM SPLIT)")
print("█"*65)

# 1. Definir el tiempo límite manualmente (ejemplo: 2 horas)
TIEMPO_MAX = 7200 

# 2. Entrenar y obtener el test_data aleatorio (el 10%)
predictor, test_data_random = train(time_limit=TIEMPO_MAX)

# 3. Evaluar usando ese mismo 10%
evaluate(predictor, val_data=test_data_random)

# 4. Predecir y guardar
predict(predictor, test_data=test_data_random, output_path="submissions/v2_random_split.csv")

print("\n ✅ Proceso completado con éxito.")



█████████████████████████████████████████████████████████████████
  AUTOGLUON — SISTEMA DE RECOMENDACIÓN YELP (RANDOM SPLIT)
█████████████████████████████████████████████████████████████████

  Cargando datos y haciendo split aleatorio...

  [⚠️] Ejecutando Split Aleatorio Baseline (90% Train / 10% Test)...
    Train: 871,005 reviews
    Test:  96,779 reviews
  Sanitizando para AutoGluon...


Preset alias specified: 'high_v150' maps to 'high_quality_v150'.
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.10
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          16
Pytorch Version:    2.11.0+cu130
CUDA Version:       13.0
GPU Memory:         GPU 0: 7.98/8.00 GB
Total GPU Memory:   Free: 7.98 GB, Allocated: 0.02 GB, Total: 8.00 GB
GPU Count:          1
Memory Avail:       14.11 GB / 31.90 GB (44.2%)
Disk Space Avail:   190.40 GB / 930.64 GB (20.5%)
Presets specified: ['high_v150']
Stack configuration (auto_stack=True): num_stack_levels=0, num_bag_folds=0, num_bag_sets=1



  Estadísticas Target Train:
    min:1  max:5  mean:3.760  std:1.479

  Iniciando entrenamiento (time_limit=7200s)...


Beginning AutoGluon training ... Time limit = 7200s
AutoGluon will save models to "c:\Users\kzzazzk\OneDrive\Documentos\MAADM\2º Cuatrimestre\2S\RECSYS\recommender-systems\competition2\models2\autogluon_1776010975_no_temporal_split"
Train Data Rows:    871005
Train Data Columns: 138
Label Column:       target
Problem Type:       regression
Preprocessing data ...
Using Feature Generators to preprocess the data ...
Fitting AutoMLPipelineFeatureGenerator...
	Available Memory:                    14542.08 MB
	Train Data (Original)  Memory Usage: 668.63 MB (4.6% of available memory)
	Stage 1 Generators:
		Fitting AsTypeFeatureGenerator...
			Note: Converting 8 features to boolean dtype as they only contain 2 unique values.
	Stage 2 Generators:
		Fitting FillNaFeatureGenerator...
	Stage 3 Generators:
		Fitting IdentityFeatureGenerator...
		Fitting CategoryFeatureGenerator...
			Fitting CategoryMemoryMinimizeFeatureGenerator...
		Fitting DatetimeFeatureGenerator...
		Fitting TextSpecialFeature


  ✓ Entrenamiento completado en 44.9 min

═════════════════════════════════════════════════════════════════
  LEADERBOARD
═════════════════════════════════════════════════════════════════
                            model  score_val     fit_time  pred_time_val
0             WeightedEnsemble_L2  -0.647039  1352.895781       0.374370
1       NeuralNetTorchRegularized  -0.647081   347.969607       0.173550
2          NeuralNetTorchStandard  -0.649391  1004.912094       0.199815
3        WeightedEnsemble_L2_FULL        NaN  1289.730418            NaN
4     NeuralNetTorchStandard_FULL        NaN   994.721795            NaN
5  NeuralNetTorchRegularized_FULL        NaN   294.994543            NaN

═════════════════════════════════════════════════════════════════
  FEATURE IMPORTANCE — top-30 (puede tardar ~2 min)
═════════════════════════════════════════════════════════════════


	258.78s	= Expected runtime (86.26s per shuffle set)
	120.57s	= Actual runtime (Completed 3 of 3 shuffle sets)


                        importance    stddev   p_value  n  p99_high   p99_low
average_stars             0.398200  0.047166  0.002322  3  0.668466  0.127934
stars_business            0.207775  0.014693  0.000831  3  0.291966  0.123583
review_useful             0.137825  0.009823  0.000844  3  0.194111  0.081538
review_cool               0.059793  0.005475  0.001392  3  0.091168  0.028418
review_funny              0.036883  0.006630  0.005299  3  0.074872 -0.001106
review_count              0.014497  0.002615  0.005337  3  0.029483 -0.000489
useful                    0.014312  0.002508  0.005042  3  0.028686 -0.000061
useful_per_review         0.009444  0.004153  0.029412  3  0.033241 -0.014352
cool                      0.005329  0.005484  0.117192  3  0.036753 -0.026095
date                      0.003278  0.002268  0.064663  3  0.016273 -0.009717
funny                     0.003174  0.001375  0.028620  3  0.011054 -0.004706
review_year               0.002752  0.001390  0.037761  3  0.010

#### INFERENCIA

In [19]:
import pandas as pd
import os
from autogluon.tabular import TabularPredictor, TabularDataset

# ─────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────
MODEL_PATH = "models2/autogluon_1776010975_no_temporal_split"   # ⚠️ cambia si usaste timestamp
TEST_PATH  = "data/test_ag.parquet"
RAW_TEST_PATH = "data/test_reviews.csv"      # ⚠️ Ajusta la ruta a tu test_reviews.csv original
OUTPUT_PATH = "submissions/submission_v2_no_temporal_split.csv"

TARGET_COL = "target"
TEXT_FEATURES = ["categories"]


# ─────────────────────────────────────────────────────────────
# SANITIZE (MISMA QUE TRAIN)
# ─────────────────────────────────────────────────────────────
def sanitize_for_autogluon(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Nullable ints → float32
    nullable_int_cols = [
        c for c in df.columns
        if str(df[c].dtype).startswith("Int")
    ]
    for c in nullable_int_cols:
        df[c] = df[c].astype("float32")

    # TEXT features
    for c in TEXT_FEATURES:
        if c in df.columns:
            df[c] = df[c].astype("string")

    # Object → category
    obj_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()
    for c in obj_cols:
        if c not in TEXT_FEATURES:
            df[c] = df[c].astype("category")

    return df


# ─────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────
def main():
    print("\n🚀 Cargando modelo...")
    predictor = TabularPredictor.load(MODEL_PATH)

    print("📂 Cargando test data...")
    test_data = TabularDataset(TEST_PATH)
    test_data = sanitize_for_autogluon(test_data)

    # Cargar solo los review_id originales para no saturar la RAM
    print("📂 Cargando review_ids originales...")
    original_test_ids = pd.read_csv(RAW_TEST_PATH, usecols=["review_id"])

    # Validar que tengan la misma cantidad de filas
    if len(original_test_ids) != len(test_data):
        print(f"⚠️ ADVERTENCIA: Las filas en {RAW_TEST_PATH} ({len(original_test_ids)}) "
              f"no coinciden con {TEST_PATH} ({len(test_data)})")

    print("🤖 Generando predicciones...")
    preds = predictor.predict(test_data, model=predictor.model_best)

    # Clipping (Yelp: 1–5 estrellas)
    preds = preds.clip(1, 5)

    print("💾 Guardando submission...")
    os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

    # Asignar el review_id real y los valores predichos
    submission = pd.DataFrame({
        "review_id": original_test_ids["review_id"],
        "stars": preds.values # .values asegura que se asigne correctamente ignorando índices de pandas
    })
    
    submission.to_csv(OUTPUT_PATH, index=False)

    print("\n✅ DONE")
    print(f"📄 Archivo: {OUTPUT_PATH}")
    print(f"📊 Stats → min={preds.min():.3f} max={preds.max():.3f} "
          f"mean={preds.mean():.3f} std={preds.std():.3f}")


if __name__ == "__main__":
    main()


🚀 Cargando modelo...
📂 Cargando test data...


Loaded data from: data/test_ag.parquet | Columns = 139 / 139 | Rows = 414765 -> 414765


📂 Cargando review_ids originales...
🤖 Generando predicciones...
💾 Guardando submission...

✅ DONE
📄 Archivo: submissions/submission_v2_no_temporal_split.csv
📊 Stats → min=1.000 max=5.000 mean=3.928 std=1.378


Test MAE: 0.6532 sin mucha diferencia vs temporal split = 0.6519

#### V3 pero sin temporal split

In [20]:
"""
train_autogluon.py  (v2 — post análisis + splits temporales)
============================================================
Cambios en esta versión:
  - Carga y ordena reviews por date_num (timestamp ascendente)
  - Hace splits temporales: 70% train, 15% val, 15% test
  - NO mezcla índices entre splits (mantiene orden cronológico)
  - Usa tuning_data del split temporal como validación interna

Dataset:
  - 1.201 categorías únicas → 'categories' como TEXT (NLP + embeddings)
  - 81 atributos → muchos son BusinessParking_* y Ambience_* (binarios)
  - 15 atributos categóricos (WiFi, NoiseLevel…) → AutoGluon label-encode
  - ~968K reviews, ~30K negocios
  - num_bag_folds=5 y num_stack_levels=1 configurados para tiempo viable

Parámetros importantes:
  - TIME_LIMIT = 14400 * 12 → 48 horas (para GPU con mucha RAM)
    Ajusta según hardware: GPU → 14400 (4h), CPU solo → 7200 (2h)
"""

import pandas as pd
import numpy as np
import os
import gc
import time

from autogluon.tabular import TabularPredictor, TabularDataset
from autogluon.common.features.feature_metadata import FeatureMetadata


# ─────────────────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────────────────
DATA_DIR   = "data2"
MODELS_DIR = "models2/autogluon3_no_temporal_split"

TRAIN_PATH  = os.path.join(DATA_DIR, "train_ag.parquet")
TEST_PATH   = os.path.join(DATA_DIR, "test_ag.parquet")

TARGET_COL  = "target"
EVAL_METRIC = "mae"

# Ajustar según hardware:
#   GPU + mucha RAM → 14400 (4h)
#   Solo CPU        → 7200  (2h)
TIME_LIMIT = 14400 * 12 # -> 48 horas
SEED       = 42


# ─────────────────────────────────────────────────────────────────────────────
# FEATURE METADATA
# Forzamos 'categories' como TEXT para activar los módulos NLP.
# El resto AutoGluon lo infiere bien a partir de los dtypes del parquet.
# ─────────────────────────────────────────────────────────────────────────────

# Columnas que deben tratarse como TEXT (NLP)
# → Solo 'categories' (string largo, multi-valor separado por comas)
TEXT_FEATURES = ["categories"]

# Las siguientes las infiere AutoGluon como 'category' a partir de dtype object:
# top_category, city, state, postal_code, elite_bucket,
# attr_WiFi, attr_NoiseLevel, attr_RestaurantsAttire, attr_Alcohol, attr_Smoking…


# ─────────────────────────────────────────────────────────────────────────────
# HIPERPARÁMETROS
# ─────────────────────────────────────────────────────────────────────────────

HYPERPARAMETERS = {
    "NN_TORCH": [
        {
            "num_epochs": 50,
            "learning_rate": 1e-3,
            "dropout_prob": 0.1,
            "weight_decay": 1e-6,
            "batch_size": 256,  # <-- ¡Bajamos a 256!
            "ag_args": {"name_suffix": "Standard"},
        },
        {
            "num_epochs": 30,
            "learning_rate": 3e-4,
            "dropout_prob": 0.3,
            "weight_decay": 1e-5,
            "batch_size": 512, # <-- ¡Bajamos a 512!
            "ag_args": {"name_suffix": "Regularized"},
        },
    ],
}

from autogluon.common.features.feature_metadata import FeatureMetadata

TEXT_FEATURES = ["categories"]

from sklearn.model_selection import train_test_split

def load_and_split_baseline(parquet_path: str):
    """
    Carga el parquet y hace exactamente el mismo split 90/10 de la baseline,
    ignorando el tiempo.
    """
    df = pd.read_parquet(parquet_path)
    
    # Quitamos la columna técnica de tiempo para que el modelo no la vea
    cols_to_drop = ["date_num"]
    df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True, errors="ignore")
    
    print("\n  [⚠️] Ejecutando Split Aleatorio Baseline (90% Train / 10% Test)...")
    
    # Redondeamos el target temporalmente para que stratify funcione sin fallos
    stratify_col = df[TARGET_COL].round().astype(int)
    
    train_data, test_data = train_test_split(
        df, 
        test_size=0.1, 
        random_state=42, 
        stratify=stratify_col
    )
    
    print(f"    Train: {len(train_data):,} reviews")
    print(f"    Test:  {len(test_data):,} reviews")
    
    return train_data.reset_index(drop=True), test_data.reset_index(drop=True)

def sanitize_for_autogluon(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # 1) Nullable integers de pandas -> float32 para evitar warnings de numpy/AutoGluon
    nullable_int_cols = [
        c for c in df.columns
        if str(df[c].dtype).startswith("Int")
    ]
    for c in nullable_int_cols:
        df[c] = df[c].astype("float32")

    # 2) Mantener el texto como texto
    for c in TEXT_FEATURES:
        if c in df.columns:
            df[c] = df[c].astype("string")

    # 3) El resto de object/string -> category para evitar detección errónea de datetime
    obj_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()
    for c in obj_cols:
        if c not in TEXT_FEATURES:
            df[c] = df[c].astype("category")

    return df


# ─────────────────────────────────────────────────────────────────────────────
# ENTRENAMIENTO
# ─────────────────────────────────────────────────────────────────────────────

def train(time_limit: int = TIME_LIMIT):
    print("\n  Cargando datos y haciendo split aleatorio...")
    
    # 1. Llamamos a la nueva función
    train_data, test_data = load_and_split_baseline(TRAIN_PATH)
    
    # 2. Aplicar sanitización
    print("  Sanitizando para AutoGluon...")
    train_data = sanitize_for_autogluon(train_data)
    test_data = sanitize_for_autogluon(test_data)
    
    print(f"\n  Estadísticas Target Train:")
    print(f"    min:{train_data[TARGET_COL].min():.0f}  "
          f"max:{train_data[TARGET_COL].max():.0f}  "
          f"mean:{train_data[TARGET_COL].mean():.3f}  "
          f"std:{train_data[TARGET_COL].std():.3f}")

    text_cols = [c for c in TEXT_FEATURES if c in train_data.columns]
    
    predictor = TabularPredictor(
        label        = TARGET_COL,
        eval_metric  = EVAL_METRIC,
        path         = MODELS_DIR,
        problem_type = "regression",
        verbosity    = 2,
    )
    
    feature_metadata = FeatureMetadata.from_df(train_data)
    feature_metadata = feature_metadata.add_special_types(
        {col: ["text"] for col in TEXT_FEATURES if col in train_data.columns}
    )

    print(f"\n  Iniciando entrenamiento (time_limit={time_limit}s)...")
    t0 = time.time()

    predictor.fit(
        train_data   = train_data,
        # OJO AQUÍ: Hemos borrado tuning_data=val_data
        # AutoGluon cogerá un % automáticamente como en tu notebook
        hyperparameters = HYPERPARAMETERS,
        time_limit   = time_limit,
        presets      = "high_v150",
        ag_args_fit  = {"num_gpus": 1},
        num_stack_levels = 0,
        num_bag_folds    = 0,
        feature_metadata = feature_metadata,
        excluded_model_types = ["KNN"],
    )

    elapsed = time.time() - t0
    print(f"\n  ✓ Entrenamiento completado en {elapsed/60:.1f} min")

    return predictor, test_data

# ─────────────────────────────────────────────────────────────────────────────
# EVALUACIÓN
# ─────────────────────────────────────────────────────────────────────────────

def evaluate(predictor: TabularPredictor, val_data: pd.DataFrame = None):
    print("\n" + "═"*65)
    print("  LEADERBOARD")
    print("═"*65)
    lb = predictor.leaderboard(silent=True)
    print(lb[["model","score_val","fit_time","pred_time_val"]].to_string())

    print("\n" + "═"*65)
    print("  FEATURE IMPORTANCE — top-30 (puede tardar ~2 min)")
    print("═"*65)
    try:
        if val_data is not None:
            data_for_fi = val_data
        else:
            data_for_fi = TabularDataset(TRAIN_PATH)
        
        fi = predictor.feature_importance(
            data             = data_for_fi,
            num_shuffle_sets = 3,
            subsample_size   = min(5000, len(data_for_fi)),
        )
        print(fi.head(30).to_string())

        # Guardar para análisis posterior
        fi.to_csv("models2/feature_importance.csv")
        print("\n  → Guardado en models/feature_importance.csv")
    except Exception as e:
        print(f"  Feature importance no disponible: {e}")

    best = predictor.model_best

    lb = predictor.leaderboard(silent=True)
    val_mae = -lb.loc[lb["model"] == best, "score_val"].values[0]
    print(f"\n  Mejor modelo : {best}")
    print(f"  MAE (val)    : {val_mae:.4f}")


# ─────────────────────────────────────────────────────────────────────────────
# PREDICCIÓN
# ─────────────────────────────────────────────────────────────────────────────

def predict(predictor: TabularPredictor,
            test_data: pd.DataFrame = None,
            output_path: str = "submissions/submission.csv"):
    print("\n" + "═"*65)
    print("  PREDICCIÓN TEST")
    print("═"*65)

    if test_data is None:
        print("  Cargando test_data desde archivo...")
        test_data = TabularDataset(TEST_PATH)
        test_data = sanitize_for_autogluon(test_data)
    
    preds = predictor.predict(test_data, model=predictor.model_best)
    # Las estrellas de Yelp son siempre [1, 5]
    preds = preds.clip(1, 5)

    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    pd.DataFrame({"stars": preds}).to_csv(output_path, index=True, index_label="index")

    print(f"  ✓ Guardado: {output_path}")
    print(f"  min={preds.min():.3f}  max={preds.max():.3f}  "
          f"mean={preds.mean():.3f}  std={preds.std():.3f}")
    return preds


def load_predictor() -> TabularPredictor:
    print(f"  Cargando predictor desde {MODELS_DIR}...")
    return TabularPredictor.load(MODELS_DIR)


# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "█"*65)
print("  AUTOGLUON — SISTEMA DE RECOMENDACIÓN YELP (RANDOM SPLIT)")
print("█"*65)

# 1. Definir el tiempo límite manualmente (ejemplo: 2 horas)
TIEMPO_MAX = 7200 

# 2. Entrenar y obtener el test_data aleatorio (el 10%)
predictor, test_data_random = train(time_limit=TIEMPO_MAX)

# 3. Evaluar usando ese mismo 10%
evaluate(predictor, val_data=test_data_random)

# 4. Predecir y guardar
predict(predictor, test_data=test_data_random, output_path="submissions/v2_random_split.csv")

print("\n ✅ Proceso completado con éxito.")


█████████████████████████████████████████████████████████████████
  AUTOGLUON — SISTEMA DE RECOMENDACIÓN YELP (RANDOM SPLIT)
█████████████████████████████████████████████████████████████████

  Cargando datos y haciendo split aleatorio...

  [⚠️] Ejecutando Split Aleatorio Baseline (90% Train / 10% Test)...
    Train: 871,005 reviews
    Test:  96,779 reviews
  Sanitizando para AutoGluon...


Preset alias specified: 'high_v150' maps to 'high_quality_v150'.
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.10
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          16
Pytorch Version:    2.11.0+cu130
CUDA Version:       13.0
GPU Memory:         GPU 0: 7.98/8.00 GB
Total GPU Memory:   Free: 7.98 GB, Allocated: 0.02 GB, Total: 8.00 GB
GPU Count:          1
Memory Avail:       12.41 GB / 31.90 GB (38.9%)
Disk Space Avail:   188.00 GB / 930.64 GB (20.2%)
Presets specified: ['high_v150']
Stack configuration (auto_stack=True): num_stack_levels=0, num_bag_folds=0, num_bag_sets=1



  Estadísticas Target Train:
    min:1  max:5  mean:3.760  std:1.479

  Iniciando entrenamiento (time_limit=7200s)...


Beginning AutoGluon training ... Time limit = 7200s
AutoGluon will save models to "c:\Users\kzzazzk\OneDrive\Documentos\MAADM\2º Cuatrimestre\2S\RECSYS\recommender-systems\competition2\models2\autogluon3_no_temporal_split"
Train Data Rows:    871005
Train Data Columns: 137
Label Column:       target
Problem Type:       regression
Preprocessing data ...
Using Feature Generators to preprocess the data ...
Fitting AutoMLPipelineFeatureGenerator...
	Available Memory:                    12821.70 MB
	Train Data (Original)  Memory Usage: 637.78 MB (5.0% of available memory)
	Stage 1 Generators:
		Fitting AsTypeFeatureGenerator...
			Note: Converting 11 features to boolean dtype as they only contain 2 unique values.
	Stage 2 Generators:
		Fitting FillNaFeatureGenerator...
	Stage 3 Generators:
		Fitting IdentityFeatureGenerator...
		Fitting CategoryFeatureGenerator...
			Fitting CategoryMemoryMinimizeFeatureGenerator...
		Fitting TextSpecialFeatureGenerator...
			Fitting BinnedFeatureGenerator.


  ✓ Entrenamiento completado en 43.2 min

═════════════════════════════════════════════════════════════════
  LEADERBOARD
═════════════════════════════════════════════════════════════════
                            model  score_val     fit_time  pred_time_val
0             WeightedEnsemble_L2  -0.882519  1357.157128       0.355289
1          NeuralNetTorchStandard  -0.882726  1017.821854       0.174622
2       NeuralNetTorchRegularized  -0.885814   339.319615       0.180667
3        WeightedEnsemble_L2_FULL        NaN  1182.716854            NaN
4     NeuralNetTorchStandard_FULL        NaN   958.685992            NaN
5  NeuralNetTorchRegularized_FULL        NaN   224.015203            NaN

═════════════════════════════════════════════════════════════════
  FEATURE IMPORTANCE — top-30 (puede tardar ~2 min)
═════════════════════════════════════════════════════════════════


	221.11s	= Expected runtime (73.7s per shuffle set)
	116.2s	= Actual runtime (Completed 3 of 3 shuffle sets)


                        importance    stddev   p_value  n  p99_high   p99_low
biz_avg_stars_at_time     0.146902  0.004727  0.000172  3  0.173986  0.119819
useful_per_review         0.135521  0.007152  0.000464  3  0.176504  0.094537
useful                    0.053813  0.011138  0.006991  3  0.117636 -0.010010
user_avg_stars_at_time    0.027987  0.012210  0.028989  3  0.097950 -0.041976
cool                      0.024781  0.002029  0.001114  3  0.036407  0.013155
review_year               0.020681  0.004726  0.008482  3  0.047760 -0.006398
delta_stars               0.010645  0.006060  0.046584  3  0.045366 -0.024077
friend_count              0.010570  0.005335  0.037714  3  0.041140 -0.019999
funny                     0.007428  0.004180  0.045677  3  0.031382 -0.016527
fans_per_review           0.005819  0.001983  0.018306  3  0.017184 -0.005546
postal_code               0.005788  0.004609  0.080803  3  0.032196 -0.020620
categories                0.004791  0.001591  0.017428  3  0.013

#### INFERENCIA

In [21]:
import pandas as pd
import os
from autogluon.tabular import TabularPredictor, TabularDataset

# ─────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────
MODEL_PATH = "models2/autogluon3_no_temporal_split"   # ⚠️ cambia si usaste timestamp
TEST_PATH  = "data2/test_ag.parquet"
RAW_TEST_PATH = "data2/test_reviews.csv"      # ⚠️ Ajusta la ruta a tu test_reviews.csv original
OUTPUT_PATH = "submissions/submission_v3_no_temporal_split.csv"

TARGET_COL = "target"
TEXT_FEATURES = ["categories"]


# ─────────────────────────────────────────────────────────────
# SANITIZE (MISMA QUE TRAIN)
# ─────────────────────────────────────────────────────────────
def sanitize_for_autogluon(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Nullable ints → float32
    nullable_int_cols = [
        c for c in df.columns
        if str(df[c].dtype).startswith("Int")
    ]
    for c in nullable_int_cols:
        df[c] = df[c].astype("float32")

    # TEXT features
    for c in TEXT_FEATURES:
        if c in df.columns:
            df[c] = df[c].astype("string")

    # Object → category
    obj_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()
    for c in obj_cols:
        if c not in TEXT_FEATURES:
            df[c] = df[c].astype("category")

    return df


# ─────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────
def main():
    print("\n🚀 Cargando modelo...")
    predictor = TabularPredictor.load(MODEL_PATH)

    print("📂 Cargando test data...")
    test_data = TabularDataset(TEST_PATH)
    test_data = sanitize_for_autogluon(test_data)

    # Cargar solo los review_id originales para no saturar la RAM
    print("📂 Cargando review_ids originales...")
    original_test_ids = pd.read_csv(RAW_TEST_PATH, usecols=["review_id"])

    # Validar que tengan la misma cantidad de filas
    if len(original_test_ids) != len(test_data):
        print(f"⚠️ ADVERTENCIA: Las filas en {RAW_TEST_PATH} ({len(original_test_ids)}) "
              f"no coinciden con {TEST_PATH} ({len(test_data)})")

    print("🤖 Generando predicciones...")
    preds = predictor.predict(test_data, model=predictor.model_best)

    # Clipping (Yelp: 1–5 estrellas)
    preds = preds.clip(1, 5)

    print("💾 Guardando submission...")
    os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

    # Asignar el review_id real y los valores predichos
    submission = pd.DataFrame({
        "review_id": original_test_ids["review_id"],
        "stars": preds.values # .values asegura que se asigne correctamente ignorando índices de pandas
    })
    
    submission.to_csv(OUTPUT_PATH, index=False)

    print("\n✅ DONE")
    print(f"📄 Archivo: {OUTPUT_PATH}")
    print(f"📊 Stats → min={preds.min():.3f} max={preds.max():.3f} "
          f"mean={preds.mean():.3f} std={preds.std():.3f}")


if __name__ == "__main__":
    main()


🚀 Cargando modelo...
📂 Cargando test data...


Loaded data from: data2/test_ag.parquet | Columns = 137 / 137 | Rows = 414765 -> 414765


📂 Cargando review_ids originales...
🤖 Generando predicciones...
💾 Guardando submission...

✅ DONE
📄 Archivo: submissions/submission_v3_no_temporal_split.csv
📊 Stats → min=1.000 max=4.998 mean=4.042 std=1.116


Test MAE: 1.3836 sin mucha diferencia vs temporal split = 1.3866

### DEFINITIVE DATA TRAINING WITH EXTREME PRESET

In [1]:
import torch
print("Versión real de PyTorch cargada:", torch.__version__)

Versión real de PyTorch cargada: 2.6.0+cu124


In [1]:
import pandas as pd
import os
from autogluon.tabular import TabularPredictor, TabularDataset

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURACIÓN
# ─────────────────────────────────────────────────────────────────────────────
DATA_DIR   = "definitive_data"
MODELS_DIR = "models2/autogluon_extreme" # Carpeta distinta para no sobreescribir

TRAIN_PATH = os.path.join(DATA_DIR, "train_preprocess1.csv")
TEST_PATH  = os.path.join(DATA_DIR, "test_preprocess1.csv")

TARGET_COL  = "target"
EVAL_METRIC = "mae"

# ─────────────────────────────────────────────────────────────────────────────
# PREPARACIÓN DE DATOS
# ─────────────────────────────────────────────────────────────────────────────

def prepare_data():
    print(f"**Cargando datos para entrenamiento simple...**")
    train_df = pd.read_csv(TRAIN_PATH)
    test_df  = pd.read_csv(TEST_PATH)
    
    # Mantenemos user_id y business_id para el comportamiento NCF
    # Eliminamos review_id por ser un identificador único de fila
    cols_to_drop = ['review_id']
    train_df = train_df.drop(columns=cols_to_drop, errors='ignore')
    test_df = test_df.drop(columns=cols_to_drop, errors='ignore')
    
    # Forzamos categorías para que NN_TORCH cree los embeddings
    for col in ['user_id', 'business_id']:
        if col in train_df.columns:
            train_df[col] = train_df[col].astype('category')
            test_df[col]  = test_df[col].astype('category')
            
    return TabularDataset(train_df), TabularDataset(test_df)

# ─────────────────────────────────────────────────────────────────────────────
# ENTRENAMIENTO (ESTILO "COLEGA")
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    train_data, test_data = prepare_data()

    predictor = TabularPredictor(
        label=TARGET_COL,
        eval_metric=EVAL_METRIC,
        path=MODELS_DIR,
        problem_type="regression",
    ).fit(
        train_data=train_data,
        presets="best", # Aunque usemos "best", los parámetros de abajo mandan
        hyperparameters={"NN_TORCH": {}}, # Solo Red Neuronal de PyTorch
        ag_args_fit={'num_gpus': 1},      # Uso de GPU
        num_bag_folds=0,                  # Sin validación cruzada (Bagging)
        num_stack_levels=0,               # Sin capas de modelos (Stacking)
        fit_weighted_ensemble=False,      # No combinar con otros modelos
        fit_full_last_level_weighted_ensemble=False,
    )

    # ─────────────────────────────────────────────────────────────────────────────
    # EVALUACIÓN Y PREDICCIÓN
    # ─────────────────────────────────────────────────────────────────────────────
    print("\n**Leaderboard (Modelo único):**")
    print(predictor.leaderboard())

    print("\n**Generando predicciones...**")
    preds = predictor.predict(test_data)
    preds = preds.clip(1, 5)

    os.makedirs("submissions", exist_ok=True)
    pd.DataFrame({"stars": preds}).to_csv("submissions/submission_extreme.csv", index=True, index_label="index")
    
    print(f"\n**✓ Proceso completado. Modelo rápido guardado en: {MODELS_DIR}**")

c:\Users\kzzazzk\OneDrive\Documentos\MAADM\2º Cuatrimestre\2S\RECSYS\recommender-systems\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


**Cargando datos para entrenamiento simple...**


Preset alias specified: 'best' maps to 'best_quality'.
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.10
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          16
Pytorch Version:    2.6.0+cu124
CUDA Version:       12.4
GPU Memory:         GPU 0: 8.00/8.00 GB
Total GPU Memory:   Free: 8.00 GB, Allocated: 0.00 GB, Total: 8.00 GB
GPU Count:          1
Memory Avail:       15.14 GB / 31.90 GB (47.5%)
Disk Space Avail:   174.02 GB / 930.64 GB (18.7%)
Presets specified: ['best']
Stack configuration (auto_stack=True): num_stack_levels=0, num_bag_folds=0, num_bag_sets=1
Beginning AutoGluon training ... Time limit = 3600s
AutoGluon will save models to "c:\Users\kzzazzk\OneDrive\Documentos\MAADM\2º Cuatrimestre\2S\RECSYS\recommender-systems\competition2\models2\autogluon_extreme"
Train Data Rows:    967784
Train Data Columns: 43
Label Column:       target
Prob


**Leaderboard (Modelo único):**
            model  score_val          eval_metric  pred_time_val    fit_time  \
0  NeuralNetTorch  -0.645463  mean_absolute_error       0.073978  1084.52091   

   pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  \
0                0.073978         1084.52091            1       True   

   fit_order  
0          1  

**Generando predicciones...**

**✓ Proceso completado. Modelo rápido guardado en: models2/autogluon_extreme**


In [ ]:
# ... (después del entrenamiento y leaderboard)

print("\n**Calculando importancia de las variables...**")

# Es recomendable usar el test_data para ver qué variables aportan más al generalizar
importance = predictor.feature_importance(train_data)

print("\n**Importancia de las variables (Feature Importance):**")
print(importance)

# Si quieres guardarlo en un CSV
importance.to_csv("feature_importance_extreme.csv")

Computing feature importance via permutation shuffling for 43 features using 5000 rows with 5 shuffle sets...



**Calculando importancia de las variables...**


	51.79s	= Expected runtime (10.36s per shuffle set)
	9.62s	= Actual runtime (Completed 5 of 5 shuffle sets)



**Importancia de las variables (Feature Importance):**
                          importance    stddev   p_value  n  p99_high  \
average_stars               0.433472  0.031131  0.000003  5  0.497572   
stars_business              0.220993  0.025414  0.000021  5  0.273321   
useful                      0.117731  0.016106  0.000041  5  0.150893   
cool                        0.096122  0.015180  0.000072  5  0.127378   
funny                       0.084883  0.015635  0.000132  5  0.117075   
useful_user                 0.064595  0.005347  0.000006  5  0.075605   
review_count                0.041211  0.005962  0.000051  5  0.053488   
funny_user                  0.040249  0.004573  0.000020  5  0.049666   
cool_user                   0.033217  0.008014  0.000377  5  0.049718   
postal_code                 0.015484  0.005963  0.002188  5  0.027761   
total_compliments           0.010176  0.002464  0.000382  5  0.015250   
date                        0.009487  0.007901  0.027469  5  0.02575

: 

In [4]:
import pandas as pd
import os
from autogluon.tabular import TabularPredictor

# ─────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────
MODEL_PATH = "models2/autogluon_extreme"
TEST_PATH  = "definitive_data/test_preprocess1.csv" 
OUTPUT_PATH = "submissions/autogluon_extreme.csv"

# ─────────────────────────────────────────────────────────────
# SANITIZE 
# ─────────────────────────────────────────────────────────────
def prepare_test_data(path):
    print(f"**Cargando y preparando test data desde {path}...**")
    df = pd.read_csv(path)
    
    # 1. Guardamos explícitamente los review_id para el CSV final
    if "review_id" in df.columns:
        ids = df["review_id"].copy()
    else:
        raise ValueError("❌ El archivo de test no tiene la columna 'review_id'. ¡Verifica tus datos!")

    # 2. Borramos SOLO 'review_id', igual que en el entrenamiento. 
    # NO borramos las fechas, el modelo las necesita.
    cols_to_drop = ['review_id']
    df_predict = df.drop(columns=cols_to_drop, errors='ignore')
    
    # 3. Forzamos tipos categóricos para los IDs
    for col in ['user_id', 'business_id']:
        if col in df_predict.columns:
            df_predict[col] = df_predict[col].astype('category')
            
    return df_predict, ids

# ─────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────
def main():
    print("\n**🚀 Cargando modelo...**")
    try:
        predictor = TabularPredictor.load(MODEL_PATH, require_version_match=False)
    except Exception as e:
        print(f"❌ Error crítico al cargar el modelo: {e}")
        return

    # 1. Preparar datos (Ahora extrae los IDs correctamente y no borra fechas)
    test_data, review_ids = prepare_test_data(TEST_PATH)

    # 2. Predecir
    print("**🤖 Generando predicciones...**")
    preds = predictor.predict(test_data)

    # 3. Clipping (Yelp: 1–5 estrellas)
    preds = preds.clip(1, 5)

    # 4. Crear DataFrame de Submission
    print("**💾 Organizando archivo de salida...**")
    submission = pd.DataFrame({
        "review_id": review_ids,
        "stars": preds.values
    })

    # 5. Guardar (index=False es clave para que no aparezca el índice de la fila)
    os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
    submission.to_csv(OUTPUT_PATH, index=False)

    print(f"\n**✅ DONE: {OUTPUT_PATH}**")

if __name__ == "__main__":
    main()


**🚀 Cargando modelo...**
**Cargando y preparando test data desde definitive_data/test_preprocess1.csv...**
**🤖 Generando predicciones...**
**💾 Organizando archivo de salida...**

**✅ DONE: submissions/autogluon_extreme.csv**


### INFERENCIA
Obtuvimos un MAE en test de 0.6390

In [1]:
%run preprocess_autogluon4.py


█████████████████████████████████████████████████████████████████
  PREPROCESADO v4 — Replica Definitivo + Interacciones
█████████████████████████████████████████████████████████████████

  Cargando negocios...
    ✓ Negocios: (30069, 6)

  Cargando usuarios...
    Shape raw: (699619, 22)
    ✓ Usuarios: (699619, 11)

═════════════════════════════════════════════════════════════════
  TRAIN
═════════════════════════════════════════════════════════════════

  Reviews: data\train_reviews.csv
    Shape: (967784, 8)
    Merge usuarios...
    Merge negocios...
    Top-20 cats: ['Restaurants', 'Food', 'Nightlife', 'Bars', 'American (Traditional)'] ... (total 20)

    ✓ Shape final    : (967784, 53)
    Columnas         : ['target', 'user_id', 'business_id', 'average_stars', 'stars_business', 'delta_stars', 'naive_pred', 'useful'] ...
    NaN totales      : 10
    ✓ Guardado: data4\train_ag.parquet (49.7 MB)

═════════════════════════════════════════════════════════════════
  TEST
══════════

In [5]:
%run train_autogluon4.py


█████████████████████████████████████████████████████████████████
  AUTOGLUON v4 — Leaky + IDs + Interacciones + Full Ensemble
█████████████████████████████████████████████████████████████████

  Cargando datos...
    Shape bruto: (967784, 53)
    Columnas: ['target', 'user_id', 'business_id', 'average_stars', 'stars_business', 'delta_stars', 'naive_pred', 'useful', 'funny', 'cool'] ...


Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.10
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          16
Pytorch Version:    2.6.0+cu124
CUDA Version:       12.4
GPU Memory:         GPU 0: 8.00/8.00 GB
Total GPU Memory:   Free: 8.00 GB, Allocated: 0.00 GB, Total: 8.00 GB
GPU Count:          1
Memory Avail:       14.00 GB / 31.90 GB (43.9%)
Disk Space Avail:   173.77 GB / 930.64 GB (18.7%)
Presets specified: ['best_quality']
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=5, num_bag_sets=1



  Split aleatorio estratificado (90/10):
    Train : 871,005 reviews
    Val   : 96,779 reviews
    Train dist: 1.0★=15.3%  2.0★=7.6%  3.0★=9.7%  4.0★=20.6%  5.0★=46.8%
    Val dist: 1.0★=15.3%  2.0★=7.6%  3.0★=9.7%  4.0★=20.6%  5.0★=46.8%

  Stats target — Train:
    min=1 max=5 mean=3.760 std=1.479

  Iniciando entrenamiento (time_limit=28800s = 8.0h)...
  Preset: best_quality  |  Stacking: L1+L2  |  Bagging: 5 folds


Beginning AutoGluon training ... Time limit = 28800s
AutoGluon will save models to "c:\Users\kzzazzk\OneDrive\Documentos\MAADM\2º Cuatrimestre\2S\RECSYS\recommender-systems\competition2\models4\autogluon_v4"
Train Data Rows:    871005
Train Data Columns: 52
Tuning Data Rows:    96779
Tuning Data Columns: 52
Label Column:       target
Problem Type:       regression
Preprocessing data ...
Using Feature Generators to preprocess the data ...
Fitting AutoMLPipelineFeatureGenerator...
	Available Memory:                    14187.30 MB
	Train Data (Original)  Memory Usage: 208.96 MB (1.5% of available memory)
	Stage 1 Generators:
		Fitting AsTypeFeatureGenerator...
			Note: Converting 23 features to boolean dtype as they only contain 2 unique values.
	Stage 2 Generators:
		Fitting FillNaFeatureGenerator...
	Stage 3 Generators:
		Fitting IdentityFeatureGenerator...
		Fitting CategoryFeatureGenerator...
			Fitting CategoryMemoryMinimizeFeatureGenerator...
	Stage 4 Generators:
		Fitting DropUniqu


  ✓ Entrenamiento completado en 245.7 min

═════════════════════════════════════════════════════════════════
  LEADERBOARD
═════════════════════════════════════════════════════════════════
                               model  score_val      fit_time  pred_time_val
0   NeuralNetTorchRegularized_BAG_L2  -0.631989  10033.307629      24.107133
1                WeightedEnsemble_L3  -0.631989  10033.484387      24.108132
2          NeuralNetTorchWide_BAG_L2  -0.632955   8648.698828      24.103658
3          NeuralNetTorchSlow_BAG_L2  -0.633403   8580.525769      24.196869
4                WeightedEnsemble_L2  -0.637889   4277.276495       6.164472
5   NeuralNetTorchRegularized_BAG_L1  -0.638148   2925.315287       3.004876
6          NeuralNetTorchWide_BAG_L1  -0.638591   1351.857418       3.158590
7          NeuralNetTorchSlow_BAG_L1  -0.639319   1262.767584       3.087862
8                    CatBoost_BAG_L2  -0.646934   7475.506968      21.499585
9             NeuralNetFastAI_BAG_L2  -0

AttributeError: 'TabularPredictor' object has no attribute 'get_model_best'

In [ ]:
import pandas as pd
import os
from autogluon.tabular import TabularPredictor

MODEL_PATH = "models4/autogluon_v4"
TEST_PATH  = "data4/test_ag.parquet"
OUTPUT_PATH = "submissions/autogluon_v4.csv"

def prepare_test_data(path):
    print(f"Cargando test desde {path}...")
    df = pd.read_parquet(path)

    if "review_id" not in df.columns:
        raise ValueError("El archivo de test no tiene review_id.")

    ids = df["review_id"].copy()
    df = df.drop(columns=["review_id"], errors="ignore")

    return df, ids

def main():
    print("Cargando modelo...")
    predictor = TabularPredictor.load(MODEL_PATH, require_version_match=False)

    test_data, review_ids = prepare_test_data(TEST_PATH)

    print("Prediciendo...")
    preds = predictor.predict(test_data).clip(1, 5)

    submission = pd.DataFrame({
        "review_id": review_ids,
        "stars": preds.values
    })

    os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
    submission.to_csv(OUTPUT_PATH, index=False)

    print(f"Guardado en {OUTPUT_PATH}")

if __name__ == "__main__":
    main()

Cargando modelo...
Cargando test desde data4/test_ag.parquet...
Prediciendo...
Guardado en submissions/autogluon_v4.csv


: 

### INFERENCIA
Obtuvimos un MAE de 0.6321 el mejor hasta ahora

## V4.1 (sin CF) - Documentacion
Este bloque documenta la iteracion `v4.1`, que es una mejora incremental de `v4` antes de meter `cf_predict`.

**`preprocess_autogluon41.py`**
- Mantiene la base fuerte de `v4` (`average_stars`, `stars_business`, `naive_pred`, interacciones).
- Corrige `friend_count` calculandolo desde `friends` antes de eliminar esa columna.
- Anade senal temporal explicita (`dayofweek`, `is_weekend`, `month_sin/cos`, `dow_sin/cos`).
- Genera datasets en `data41/train_ag.parquet` y `data41/test_ag.parquet`.

**`train_autogluon41.py`**
- Entrena AutoGluon sobre `data41/` con setup parecido a `v4` para comparar manzanas con manzanas.
- Elimina dependencia de `categories` como texto (en `v4.1` ya va OHE).
- Incluye un `evaluate()` compatible con versiones donde no existe `get_model_best()`.
- Guarda en `models41/autogluon_v41`.

Uso recomendado para reproducir `v4.1`:
- `%run preprocess_autogluon41.py`
- `%run train_autogluon41.py`


## PREPROCESSING DEFINITIVO + CF_PREDICT
Este bloque consolida lo que aprendimos en v1-v4 y deja el dataset final listo para AutoGluon.

- Conserva la base fuerte de `v4`: `average_stars`, `stars_business`, `naive_pred` y las interacciones derivadas.
- Corrige `friend_count` calculandolo de verdad desde `friends` antes de eliminar la columna.
- Recupera senal temporal util con `dayofweek`, `is_weekend` y codificaciones ciclicas de mes/dia de semana.
- Genera `cf_predict` con `SVD++` de `surprise`, usando OOF en train y fit completo para test.
- Anade `cf_minus_naive`, `cf_times_naive`, `cf_known_user` y `cf_known_business` para que AutoGluon vea mejor la cobertura CF.

La idea no es cambiar todo el pipeline, sino quedarnos con las senales que ya vimos que aportan y sumar una feature colaborativa controlada.


In [1]:
%run preprocess_autogluon_definitive.py



█████████████████████████████████████████████████████████████████
  PREPROCESADO DEFINITIVO — v4 + tiempo + friend_count + cf_predict
█████████████████████████████████████████████████████████████████

  Construyendo cf_predict con SVD++...
    Fold 1/5...
 processing epoch 0
 processing epoch 1
 processing epoch 2
 processing epoch 3
 processing epoch 4
 processing epoch 5
 processing epoch 6
 processing epoch 7
 processing epoch 8
 processing epoch 9
 processing epoch 10
 processing epoch 11
    Fold 2/5...
 processing epoch 0
 processing epoch 1
 processing epoch 2
 processing epoch 3
 processing epoch 4
 processing epoch 5
 processing epoch 6
 processing epoch 7
 processing epoch 8
 processing epoch 9
 processing epoch 10
 processing epoch 11
    Fold 3/5...
 processing epoch 0
 processing epoch 1
 processing epoch 2
 processing epoch 3
 processing epoch 4
 processing epoch 5
 processing epoch 6
 processing epoch 7
 processing epoch 8
 processing epoch 9
 processing epoch 10
 proce

In [2]:
%run train_autogluon_definitive.py


c:\Users\kzzazzk\OneDrive\Documentos\MAADM\2º Cuatrimestre\2S\RECSYS\recommender-systems\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



  Cargando datos...
    Shape bruto: (967784, 66)
    Columnas: ['target', 'user_id', 'business_id', 'average_stars', 'stars_business', 'naive_pred', 'cf_predict', 'cf_minus_naive', 'friend_count', 'dayofweek', 'is_weekend', 'review_id', 'useful', 'funny'] ...


Verbosity: 2 (Standard Logging)



  Split aleatorio estratificado (90/10):
    Train : 871,005 reviews
    Val   : 96,779 reviews
    Train dist: 1.0*=15.3%  2.0*=7.6%  3.0*=9.7%  4.0*=20.6%  5.0*=46.8%
    Val dist: 1.0*=15.3%  2.0*=7.6%  3.0*=9.7%  4.0*=20.6%  5.0*=46.8%

  Iniciando entrenamiento (time_limit=28800s = 8.0h)...
  Preset: best_quality  |  Stacking: L1+L2  |  Bagging: 5 folds


=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.10
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          16
Pytorch Version:    2.6.0+cu124
CUDA Version:       12.4
GPU Memory:         GPU 0: 8.00/8.00 GB
Total GPU Memory:   Free: 8.00 GB, Allocated: 0.00 GB, Total: 8.00 GB
GPU Count:          1
Memory Avail:       8.40 GB / 31.90 GB (26.3%)
Disk Space Avail:   168.14 GB / 930.64 GB (18.1%)
Presets specified: ['best_quality']
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=5, num_bag_sets=1
Beginning AutoGluon training ... Time limit = 28800s
AutoGluon will save models to "c:\Users\kzzazzk\OneDrive\Documentos\MAADM\2º Cuatrimestre\2S\RECSYS\recommender-systems\competition2\models_definitive\autogluon_definitive_cf"
Train Data Rows:    871005
Train Data Columns: 64
Tuning Data Rows:    96779
Tuning Data Columns: 64
Label Column:       target
Problem Type:   

KeyboardInterrupt: 

In [1]:
%run train_autogluon_definitive.py --cf-mode flags_only --model-mode nn_only --time-limit 7200 --presets best_quality


c:\Users\kzzazzk\OneDrive\Documentos\MAADM\2º Cuatrimestre\2S\RECSYS\recommender-systems\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



  Cargando datos...
    Shape bruto: (967784, 66)
    Columnas: ['target', 'user_id', 'business_id', 'average_stars', 'stars_business', 'naive_pred', 'cf_predict', 'cf_minus_naive', 'friend_count', 'dayofweek', 'is_weekend', 'review_id', 'useful', 'funny'] ...
    CF_FEATURE_MODE: flags_only
    MODEL_MODE     : nn_only
    Shape tras modo CF: (967784, 62)


Verbosity: 2 (Standard Logging)



  Split aleatorio estratificado (90/10):
    Train : 871,005 reviews
    Val   : 96,779 reviews
    Train dist: 1.0*=15.3%  2.0*=7.6%  3.0*=9.7%  4.0*=20.6%  5.0*=46.8%
    Val dist: 1.0*=15.3%  2.0*=7.6%  3.0*=9.7%  4.0*=20.6%  5.0*=46.8%
    Modelo de CF: flags_only
    Modelo base : nn_only
    Guardado en : models_definitive\autogluon_definitive_cf_flags_only_nn_only

  Iniciando entrenamiento (time_limit=7200s = 2.0h)...
  Preset: best_quality  |  Stacking: L1+L2  |  Bagging: 5 folds


=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.10
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          16
Pytorch Version:    2.6.0+cu124
CUDA Version:       12.4
GPU Memory:         GPU 0: 8.00/8.00 GB
Total GPU Memory:   Free: 8.00 GB, Allocated: 0.00 GB, Total: 8.00 GB
GPU Count:          1
Memory Avail:       18.56 GB / 31.90 GB (58.2%)
Disk Space Avail:   171.32 GB / 930.64 GB (18.4%)
Presets specified: ['best_quality']
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=5, num_bag_sets=1
Beginning AutoGluon training ... Time limit = 7200s
AutoGluon will save models to "c:\Users\kzzazzk\OneDrive\Documentos\MAADM\2º Cuatrimestre\2S\RECSYS\recommender-systems\competition2\models_definitive\autogluon_definitive_cf_flags_only_nn_only"
Train Data Rows:    871005
Train Data Columns: 61
Tuning Data Rows:    96779
Tuning Data Columns: 61
Label Column:       targ

KeyboardInterrupt: 

In [2]:
%run train_autogluon_definitive.py --cf-mode none --model-mode nn_only --time-limit 7200 --presets best_quality


  Cargando datos...
    Shape bruto: (967784, 66)
    Columnas: ['target', 'user_id', 'business_id', 'average_stars', 'stars_business', 'naive_pred', 'cf_predict', 'cf_minus_naive', 'friend_count', 'dayofweek', 'is_weekend', 'review_id', 'useful', 'funny'] ...
    CF_FEATURE_MODE: none
    MODEL_MODE     : nn_only
    Shape tras modo CF: (967784, 60)


Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.10
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          16
Pytorch Version:    2.6.0+cu124
CUDA Version:       12.4
GPU Memory:         GPU 0: 7.99/8.00 GB
Total GPU Memory:   Free: 7.99 GB, Allocated: 0.01 GB, Total: 8.00 GB
GPU Count:          1
Memory Avail:       13.25 GB / 31.90 GB (41.6%)
Disk Space Avail:   168.40 GB / 930.64 GB (18.1%)
Presets specified: ['best_quality']
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=5, num_bag_sets=1



  Split aleatorio estratificado (90/10):
    Train : 871,005 reviews
    Val   : 96,779 reviews
    Train dist: 1.0*=15.3%  2.0*=7.6%  3.0*=9.7%  4.0*=20.6%  5.0*=46.8%
    Val dist: 1.0*=15.3%  2.0*=7.6%  3.0*=9.7%  4.0*=20.6%  5.0*=46.8%
    Modelo de CF: none
    Modelo base : nn_only
    Guardado en : models_definitive\autogluon_definitive_cf_none_nn_only

  Iniciando entrenamiento (time_limit=7200s = 2.0h)...
  Preset: best_quality  |  Stacking: L1+L2  |  Bagging: 5 folds


Beginning AutoGluon training ... Time limit = 7200s
AutoGluon will save models to "c:\Users\kzzazzk\OneDrive\Documentos\MAADM\2º Cuatrimestre\2S\RECSYS\recommender-systems\competition2\models_definitive\autogluon_definitive_cf_none_nn_only"
Train Data Rows:    871005
Train Data Columns: 59
Tuning Data Rows:    96779
Tuning Data Columns: 59
Label Column:       target
Problem Type:       regression
Preprocessing data ...
Using Feature Generators to preprocess the data ...
Fitting AutoMLPipelineFeatureGenerator...
	Available Memory:                    13385.81 MB
	Train Data (Original)  Memory Usage: 232.04 MB (1.7% of available memory)
	Stage 1 Generators:
		Fitting AsTypeFeatureGenerator...
			Note: Converting 23 features to boolean dtype as they only contain 2 unique values.
	Stage 2 Generators:
		Fitting FillNaFeatureGenerator...
	Stage 3 Generators:
		Fitting IdentityFeatureGenerator...
		Fitting CategoryFeatureGenerator...
			Fitting CategoryMemoryMinimizeFeatureGenerator...
	Stage 


  Entrenamiento completado en 119.9 min

═════════════════════════════════════════════════════════════════
  EVALUACION
═════════════════════════════════════════════════════════════════
                           model  score_val    fit_time  pred_time_val
       NeuralNetTorchWide_BAG_L1  -0.640544 1763.316020       3.589692
             WeightedEnsemble_L2  -0.640544 1763.373213       3.591692
             WeightedEnsemble_L3  -0.640544 1763.421857       3.590692
       NeuralNetTorchWide_BAG_L2  -0.643961 4252.566619      14.185772
NeuralNetTorchRegularized_BAG_L2  -0.645486 3298.064252      14.039181
NeuralNetTorchRegularized_BAG_L1  -0.647309  734.551689       3.426570
       NeuralNetTorchSlow_BAG_L2  -0.666142 2823.869145      14.045296
       NeuralNetTorchSlow_BAG_L1  -0.682098  168.852915       3.399981

  Mejor modelo : WeightedEnsemble_L2
  Validacion   : {'mean_absolute_error': -0.6405438383471632, 'root_mean_squared_error': -1.0849488621965608, 'mean_squared_error': -1.1

In [3]:
import os
import pandas as pd
from autogluon.tabular import TabularPredictor

MODEL_PATH = "models_definitive/autogluon_definitive_cf"
TEST_PATH = "data_definitive_cf/test_ag.parquet"
OUTPUT_PATH = "submissions/autogluon_definitive_cf.csv"

print("Cargando modelo...")
predictor = TabularPredictor.load(MODEL_PATH, require_version_match=False)

print(f"Cargando test desde {TEST_PATH}...")
test_df = pd.read_parquet(TEST_PATH)
review_ids = test_df["review_id"].copy()
test_df = test_df.drop(columns=["review_id"], errors="ignore")

print("Prediciendo...")
preds = predictor.predict(test_df).clip(1, 5)

submission = pd.DataFrame({"review_id": review_ids, "stars": preds.values})
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
submission.to_csv(OUTPUT_PATH, index=False)
print(f"Guardado en {OUTPUT_PATH}")


Cargando modelo...
Cargando test desde data_definitive_cf/test_ag.parquet...
Prediciendo...
Guardado en submissions/autogluon_definitive_cf.csv


### RESULTADO DEFINITIVO
Anota aqui el MAE del pipeline con `cf_predict`.
